# 4\.2\.2 Construct Multimodal and Spatial Context Features

Notebook 4.2.1 described each forecast target using its own recent history, comparable prior periods, seasonal structure, and calendar context. This notebook widens that view.

A mobility forecast may also benefit from knowing what **other transportation modes** were doing in the same place and what was happening in **transportation-connected Taxi Zones** when the forecast was issued. For example, Taxi demand may be easier to predict when the model can also see recent FHVHV or Subway conditions, while a local speed forecast may benefit from mobility conditions in connected parts of the city.

The feature set stays intentionally compact. Rather than generating a large library of possible interactions, this notebook starts with a small number of understandable signals that describe:

1. **Multimodal context** — what other mobility measures were doing at or before the forecast origin.
2. **Spatial context** — what transportation-connected and broader surrounding locations were doing.
3. **Static geographic context** — stable Taxi Zone characteristics such as borough, CBD relationship, and transportation connectivity.

Every dynamic feature follows the same information boundary used in the forecasting design: values observed **at or before the forecast origin** may be used; mobility occurring after that point may not.

The final context layer preserves the same origin structure established in 4.2.1:

> **Modeling Track × Forecast Sequence × Origin observation_sequence_id**

Multimodal and spatial conditions are first calculated once for each physical Taxi Zone × origin state, then attached to the individual forecast sequences that share that origin. This avoids needlessly calculating the same surrounding transportation conditions five times for five different target metrics.

The later no-congestion-pricing simulation requires a stricter boundary. After January 4, 2025, actual post-CP mobility cannot update the hypothetical no-CP transportation state. Context derived from the five mobility measures being forecast can later be rebuilt from recursively generated predictions. Other changing mobility measures cannot simply be filled with the actual post-CP values.

Weather is handled separately in Notebook 4.2.3 because its forecasting-availability assumptions differ from mobility data.

## Notebook Setup

Define the shared forecasting contract, paths, rebuild controls, feature groups, chart settings, and helper functions used throughout the notebook.

The notebook follows the same grain and modeling-track definitions established upstream. Reusable feature surfaces are written as Parquet intermediates so completed sections do not need to be rebuilt unnecessarily.

In [1]:
# 4.2.2.0A — Define imports, globals, paths, toggles, and feature contracts

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display
from project_branding import BRAND_COLORS, BRAND_COLOR_SEQUENCE


# ---------------------------------------------------------------------
# Display and chart defaults
# ---------------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

CHART_WIDTH = 920
CHART_HEIGHT = 520


# ---------------------------------------------------------------------
# Notebook behavior
# ---------------------------------------------------------------------

WRITE_INTERMEDIATES = True
WRITE_FINAL_OUTPUTS = True

REBUILD_MULTIMODAL_CANDIDATE_STATE = False
REBUILD_SPATIAL_CONTEXT = False
REBUILD_FINAL_CONTEXT_LAYER = False


# ---------------------------------------------------------------------
# Project directories
# ---------------------------------------------------------------------

PIPELINE_DIR = Path("pipeline_data")

INPUT_131_DIR = PIPELINE_DIR / "1.3.1.final_tables"
INPUT_421_DIR = PIPELINE_DIR / "4.2.1.final_tables"

INTERMEDIATE_DIR = PIPELINE_DIR / "4.2.2.intermediate_tables"
FINAL_DIR = PIPELINE_DIR / "4.2.2.final_tables"

INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Authoritative upstream inputs
# ---------------------------------------------------------------------

MOBILITY_PANEL_PATH = (
    INPUT_131_DIR
    / "analysis_ready_mobility_panel.parquet"
)

# Current authoritative Taxi Zone reference. This already carries the static
# geographic and policy fields attached later in 4.2.2.3.
TAXI_ZONE_SPATIAL_CONTEXT_PATH = (
    INPUT_131_DIR
    / "nyc_taxi_zones_harmonized.parquet"
)

# Authoritative Taxi Zone adjacency / transportation-connectivity reference.
# Keep the table's upstream relationship schema intact; 4.2.2.3 will inspect
# and use those existing fields rather than assuming a source/neighbor layout.
TAXI_ZONE_CONNECTIVITY_PATH = (
    INPUT_131_DIR
    / "taxi_zone_connectivity.parquet"
)

ORIGIN_FEATURE_PATHS = {
    "post_cp_primary":
        INPUT_421_DIR / "origin_feature_surface__post_cp_primary.parquet",

    "full_history_regime_aware":
        INPUT_421_DIR / "origin_feature_surface__full_history_regime_aware.parquet",

    "pre_cp_reference":
        INPUT_421_DIR / "origin_feature_surface__pre_cp_reference.parquet",
}

FORECAST_FEATURE_CATALOG_PATH = (
    INPUT_421_DIR
    / "forecast_feature_catalog.parquet"
)

FORECAST_TARGET_CONTEXT_PATH = (
    INPUT_421_DIR
    / "forecast_target_calendar_context.parquet"
)

FINAL_421_QA_PATH = (
    INPUT_421_DIR
    / "final_feature_contract_qa.parquet"
)

PRE_CP_COUNTERFACTUAL_CALENDAR_PATH = (
    INPUT_421_DIR
    / "pre_cp_counterfactual_calendar.parquet"
)

PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH = (
    INPUT_421_DIR
    / "pre_cp_recursive_feature_contract.parquet"
)


# ---------------------------------------------------------------------
# 4.2.2 intermediate and final paths
# ---------------------------------------------------------------------

MULTIMODAL_CANDIDATE_STATE_PATH = (
    INTERMEDIATE_DIR
    / "multimodal_origin_candidate_state.parquet"
)

MULTIMODAL_CANDIDATE_CATALOG_PATH = (
    INTERMEDIATE_DIR
    / "multimodal_candidate_feature_catalog.parquet"
)

FINAL_CONTEXT_SURFACE_PATHS = {
    track:
        FINAL_DIR / f"origin_context_surface__{track}.parquet"
    for track in ORIGIN_FEATURE_PATHS
}

FINAL_CONTEXT_CATALOG_PATH = (
    FINAL_DIR
    / "forecast_context_feature_catalog.parquet"
)

PRE_CP_CONTEXT_RUNTIME_CONTRACT_PATH = (
    FINAL_DIR
    / "pre_cp_context_runtime_contract.parquet"
)

FINAL_CONTEXT_QA_PATH = (
    FINAL_DIR
    / "final_context_contract_qa.parquet"
)

FINAL_CONTEXT_MANIFEST_PATH = (
    FINAL_DIR
    / "forecast_context_manifest.parquet"
)


# ---------------------------------------------------------------------
# Time and modeling-track contract
# ---------------------------------------------------------------------

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")

CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")
PRE_CP_FREEZE_DATE = pd.Timestamp("2025-01-04")

MODELING_TRACKS = [
    "post_cp_primary",
    "full_history_regime_aware",
    "pre_cp_reference",
]

TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
    "pre_cp_reference": "Pre-CP Reference",
}


# ---------------------------------------------------------------------
# Mobility feature universe
# ---------------------------------------------------------------------

# These five measures are forecast targets. Their future counterfactual state
# can therefore be rebuilt recursively from model-generated no-CP predictions.
FORECAST_TARGETS = [
    "taxi_trip_count",
    "taxi_avg_trip_speed",
    "fhvhv_trip_count",
    "fhvhv_avg_trip_speed",
    "subway_ridership",
]

# These five measures can provide useful context to downstream Tree and NN
# models, but they are not themselves part of the current forecasting target
# set. That distinction matters later for the frozen no-CP simulation.
AUXILIARY_CONTEXT_METRICS = [
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_duration",
    "subway_transfers",
    "bus_trip_count",
    "avg_bus_speed",
]

MOBILITY_CONTEXT_METRICS = (
    FORECAST_TARGETS
    + AUXILIARY_CONTEXT_METRICS
)

METRIC_LABELS = {
    "taxi_trip_count": "Taxi trips",
    "taxi_avg_trip_speed": "Taxi speed",
    "taxi_avg_trip_duration": "Taxi trip duration",
    "fhvhv_trip_count": "FHVHV trips",
    "fhvhv_avg_trip_speed": "FHVHV speed",
    "fhvhv_avg_trip_duration": "FHVHV trip duration",
    "subway_ridership": "Subway ridership",
    "subway_transfers": "Subway transfers",
    "bus_trip_count": "Bus trips",
    "avg_bus_speed": "Bus speed",
}


# ---------------------------------------------------------------------
# Canonical analytical keys
# ---------------------------------------------------------------------

# One row in a 4.2.1 origin surface is uniquely identified by:
# Modeling Track × Forecast Sequence × Origin observation_sequence_id.
#
# `metric` remains an important descriptive column, but forecast_sequence_id
# already identifies the Taxi Zone × metric sequence, so metric is not part of
# the declared grain.
ORIGIN_JOIN_KEYS = [
    "modeling_track",
    "forecast_sequence_id",
    "observation_sequence_id",
]

# One physical transportation state is shared by all forecast targets in the
# same Taxi Zone at the same origin.
PHYSICAL_ORIGIN_KEYS = [
    "modeling_track",
    "taxi_zone_id",
    "observation_sequence_id",
]

# Canonical observed mobility grain used for origin-state lookup.
CANONICAL_MOBILITY_KEYS = [
    "taxi_zone_id",
    "observation_sequence_id",
]

# Authoritative 4.2.1 target-calendar grain.
TARGET_CONTEXT_KEYS = [
    "modeling_track",
    "origin_observation_sequence_id",
    "horizon",
]


# ---------------------------------------------------------------------
# Verified CP sequence boundary
# ---------------------------------------------------------------------

# These values are independently checked again against the canonical calendar
# in 4.2.2.1F rather than trusted only as constants.
EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID = 3675
EXPECTED_CP_FIRST_SEQUENCE_ID = 3676

# ---------------------------------------------------------------------
# Static Taxi Zone fields planned for 4.2.2.4
# ---------------------------------------------------------------------

# The authoritative 1.3.1 spatial reference uses `location_id`. It will be
# renamed once to `taxi_zone_id` when loaded so the remainder of the forecasting
# notebook can use the canonical mobility-panel naming convention.
STATIC_ZONE_SOURCE_COLUMNS = [
    "location_id",
    "canonical_location_id",
    "zone",
    "borough",
    "is_cbd_zone",
    "is_cbd_gateway_zone",
    "is_cbd_adjacent_zone",
    "cbd_spatial_category",
    "distance_to_cbd_miles",
    "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

# Standardized names after the upstream boundary is crossed.
STATIC_ZONE_FEATURE_COLUMNS = [
    "taxi_zone_id",
    "canonical_location_id",
    "zone",
    "borough",
    "is_cbd_zone",
    "is_cbd_gateway_zone",
    "is_cbd_adjacent_zone",
    "cbd_spatial_category",
    "distance_to_cbd_miles",
    "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

Define a small set of reusable helpers for loading upstream artifacts, protecting analytical grains, formatting QA output, and keeping chart styling consistent.

Schema validation is concentrated at upstream notebook boundaries. DataFrames created inside 4.2.2 are instead protected by explicit merge validation, uniqueness checks, and leakage-specific QA.

In [2]:
# 4.2.2.0B — Define shared loading, QA, and chart helpers

def require_file(path, name):
    """Fail clearly when an authoritative upstream artifact is unavailable."""
    if not path.exists():
        raise FileNotFoundError(
            f"{name} was not found at: {path}"
        )


def parquet_columns(path):
    """Return top-level column names without materializing the full Parquet."""
    require_file(path, "Parquet input")

    return list(
        pq.ParquetFile(path).schema_arrow.names
    )


def require_parquet_columns(path, required_columns, name):
    """
    Validate the schema of an upstream Parquet artifact before loading it.

    This protects notebook boundaries without adding repetitive schema checks
    to DataFrames whose structures are created locally in 4.2.2.
    """
    available = set(
        parquet_columns(path)
    )

    missing = sorted(
        set(required_columns).difference(available)
    )

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )


def validate_unique_key(df, key_columns, name):
    """Raise when a DataFrame violates its declared analytical grain."""
    duplicate_rows = int(
        df.duplicated(key_columns).sum()
    )

    if duplicate_rows:
        raise ValueError(
            f"{name} contains {duplicate_rows:,} duplicate rows "
            f"for key {key_columns}."
        )


def qa_row(check, passed, detail):
    """Create one compact reader-facing QA record."""
    return {
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "detail": detail,
    }


def parquet_inventory_row(name, path, role):
    """Summarize one Parquet artifact without loading the full dataset."""
    require_file(path, name)

    parquet_file = pq.ParquetFile(path)

    return {
        "input": name,
        "role": role,
        "rows": int(parquet_file.metadata.num_rows),
        "columns": len(parquet_file.schema_arrow.names),
    }


def load_or_build_parquet(path, rebuild, builder):
    """
    Load a reusable intermediate artifact or rebuild it when requested.

    Existing outputs are reused by default. New outputs are written atomically
    so an interrupted write cannot leave a partially completed Parquet behind.
    """
    if path.exists() and not rebuild:
        return pd.read_parquet(path)

    result = builder()

    if WRITE_INTERMEDIATES:
        temp_path = path.with_suffix(".tmp.parquet")

        result.to_parquet(
            temp_path,
            index=False,
        )

        temp_path.replace(path)

    return result


def multimodal_level_column(metric):
    """Return the standardized at-origin multimodal level feature name."""
    return f"mm_{metric}_origin_value"


def multimodal_delta_column(metric):
    """Return the standardized exact prior-step change feature name."""
    return f"mm_{metric}_seq_delta_1"


def apply_chart_layout(
    fig,
    *,
    title,
    x_title=None,
    y_title=None,
    height=CHART_HEIGHT,
):
    """
    Apply compact project-consistent Plotly styling.

    Chart dimensions remain below the project maximum and hover precision is
    controlled explicitly within each trace.
    """
    fig.update_layout(
        title=title,
        width=CHART_WIDTH,
        height=height,
        template="plotly_white",
        margin=dict(l=70, r=30, t=75, b=65),
        font=dict(size=12),
        xaxis_title=x_title,
        yaxis_title=y_title,
        legend_title_text="",
    )

    return fig

# 4.2.2.1 Load and Validate Forecasting Context Inputs

Before constructing new predictors, load the completed forecasting handoff together with the mobility and spatial sources needed for this notebook.

This section does not make any feature-selection decisions. Its job is to confirm that 4.2.2 begins from the same forecasting populations, sequence calendar, policy boundaries, and origin definitions already established upstream.

The large 4.2.1 feature surfaces are read only for the columns needed to recover the forecast origins. Multimodal and spatial conditions can then be calculated once for each:

> **Modeling Track × Taxi Zone × Origin observation_sequence_id**

The resulting context can later be attached to each target-specific Forecast Sequence without calculating the same surrounding transportation state repeatedly.

**Inventory authoritative forecasting and context inputs.** Start with a compact inventory of the upstream handoff. This confirms that the expected inputs are present and gives a quick sense of their size before the larger tables are loaded.

In [3]:
# 4.2.2.1A — Inventory authoritative forecasting and context inputs

input_specs = []

for track, path in ORIGIN_FEATURE_PATHS.items():
    input_specs.append(
        (
            f"4.2.1 origin surface · {TRACK_LABELS[track]}",
            path,
            "Forecast-origin backbone",
        )
    )

input_specs.extend([
    (
        "4.2.1 feature catalog",
        FORECAST_FEATURE_CATALOG_PATH,
        "Temporal and seasonal feature definitions",
    ),
    (
        "4.2.1 target context",
        FORECAST_TARGET_CONTEXT_PATH,
        "Horizon-specific target calendar",
    ),
    (
        "4.2.1 final QA",
        FINAL_421_QA_PATH,
        "Completed upstream feature-contract QA",
    ),
    (
        "Pre-CP counterfactual calendar",
        PRE_CP_COUNTERFACTUAL_CALENDAR_PATH,
        "Post-freeze recursive application calendar",
    ),
    (
        "Pre-CP recursive feature contract",
        PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH,
        "Existing recursive-feature requirements",
    ),
    (
        "Analysis-ready mobility panel",
        MOBILITY_PANEL_PATH,
        "Canonical multimodal mobility source",
    ),
    (
        "Taxi Zone spatial context",
        TAXI_ZONE_SPATIAL_CONTEXT_PATH,
        "Static geography and policy attributes",
    ),
    (
        "Taxi Zone connectivity",
        TAXI_ZONE_CONNECTIVITY_PATH,
        "Transportation-aware zone relationships",
    ),
])

input_inventory_df = pd.DataFrame([
    parquet_inventory_row(
        name,
        path,
        role,
    )
    for name, path, role in input_specs
])

input_inventory_df["rows"] = (
    input_inventory_df["rows"]
    .astype("int64")
)

display(input_inventory_df)

,input,role,rows,columns
0,4.2.1 origin surface · Post-CP Primary,Forecast-origin backbone,2497432,39
1,4.2.1 origin surface · Full-History Regime-Aware,Forecast-origin backbone,6231379,39
2,4.2.1 origin surface · Pre-CP Reference,Forecast-origin backbone,3818325,39
3,4.2.1 feature catalog,Temporal and seasonal feature definitions,37,8
4,4.2.1 target context,Horizon-specific target calendar,35496,42
5,4.2.1 final QA,Completed upstream feature-contract QA,24,3
6,Pre-CP counterfactual calendar,Post-freeze recursive application calendar,6760,23
7,Pre-CP recursive feature contract,Existing recursive-feature requirements,17,10
8,Analysis-ready mobility panel,Canonical multimodal mobility source,1559590,42
9,Taxi Zone spatial context,Static geography and policy attributes,264,19


Findings\. The required forecasting handoff is complete at the expected scales\. The three 4\.2\.1 origin surfaces are present with 39 columns each, along with the 35,496\-row horizon\-specific target context, the 37\-feature temporal/seasonal catalog, and the completed 24\-row upstream QA record\. The new context inputs are also available at their native grains: 1,559,590 canonical mobility rows, 264 static Taxi Zone records, and 876 transportation\-connectivity relationships\.

**Recover the physical forecast-origin registry.** Load the completed 4.2.1 origin surfaces and recover the unique physical forecast origins. The original surfaces remain target-specific because each Taxi Zone contributes separate sequences for the five forecast metrics. The surrounding multimodal and spatial state, however, is the same physical transportation state for all five targets at a given Taxi Zone and origin. Collapsing to that shared origin here keeps the new context layer much smaller while preserving the ability to join it back to every forecast sequence later.

In [4]:
# 4.2.2.1B — Recover the physical forecast-origin registry

ORIGIN_REQUIRED_COLUMNS = [
    "modeling_track",
    "forecast_sequence_id",
    "taxi_zone_id",
    "metric",
    "observation_sequence_id",
    "origin_date",
    "origin_daypart",
    "origin_daypart_order",
    "origin_temporal_bucket",
    "origin_temporal_bucket_order",
    "origin_pre_post_cp",
]

origin_surface_summary_records = []
physical_origin_parts = []

for track, path in ORIGIN_FEATURE_PATHS.items():
    require_parquet_columns(
        path,
        ORIGIN_REQUIRED_COLUMNS,
        f"{TRACK_LABELS[track]} origin surface",
    )

    # Only origin identity/calendar columns are required here. The actual
    # 4.2.1 predictors are not duplicated into this notebook yet.
    surface = pd.read_parquet(
        path,
        columns=ORIGIN_REQUIRED_COLUMNS,
    )

    if not surface["modeling_track"].eq(track).all():
        raise ValueError(
            f"{TRACK_LABELS[track]} contains rows from another modeling track."
        )

    duplicate_origin_keys = int(
        surface.duplicated(
            ORIGIN_JOIN_KEYS
        ).sum()
    )

    if duplicate_origin_keys:
        raise ValueError(
            f"{TRACK_LABELS[track]} contains "
            f"{duplicate_origin_keys:,} duplicate sequence-origin keys."
        )

    surface["origin_date"] = pd.to_datetime(
        surface["origin_date"]
    )

    # Several target sequences share one physical Taxi Zone × origin.
    physical = (
        surface[
            [
                "modeling_track",
                "taxi_zone_id",
                "observation_sequence_id",
                "origin_date",
                "origin_daypart",
                "origin_daypart_order",
                "origin_temporal_bucket",
                "origin_temporal_bucket_order",
                "origin_pre_post_cp",
            ]
        ]
        .drop_duplicates()
        .copy()
    )

    validate_unique_key(
        physical,
        PHYSICAL_ORIGIN_KEYS,
        f"{TRACK_LABELS[track]} physical origin registry",
    )

    origin_surface_summary_records.append({
        "modeling_track": track,
        "forecast_sequence_origin_rows": len(surface),
        "forecast_sequences": surface["forecast_sequence_id"].nunique(),
        "physical_origins": len(physical),
        "first_origin_date": surface["origin_date"].min(),
        "last_origin_date": surface["origin_date"].max(),
    })

    physical_origin_parts.append(
        physical
    )


origin_surface_summary_df = pd.DataFrame(
    origin_surface_summary_records
)

forecast_physical_origin_df = (
    pd.concat(
        physical_origin_parts,
        ignore_index=True,
    )
    .sort_values(
        [
            "modeling_track",
            "taxi_zone_id",
            "observation_sequence_id",
        ]
    )
    .reset_index(drop=True)
)

validate_unique_key(
    forecast_physical_origin_df,
    PHYSICAL_ORIGIN_KEYS,
    "Combined physical forecast-origin registry",
)


# ---------------------------------------------------------------------
# Load the lightweight 4.2.1 contracts needed later in this notebook.
#
# Only require fields that:
#   1. are verified to exist in the authoritative artifact, and
#   2. are actually consumed by 4.2.2.
#
# The 4.2.1 target-context artifact contains some suffixed intermediate
# target-calendar columns. We deliberately do not depend on those fields here.
# ---------------------------------------------------------------------

require_parquet_columns(
    FORECAST_FEATURE_CATALOG_PATH,
    ["feature_name"],
    "4.2.1 feature catalog",
)

require_parquet_columns(
    FORECAST_TARGET_CONTEXT_PATH,
    [
        "modeling_track",
        "stage",
        "origin_observation_sequence_id",
        "horizon",
        "target_observation_sequence_id",
        "origin_date",
        "origin_daypart",
        "origin_temporal_bucket",
        "target_date",
        "target_daypart",
    ],
    "4.2.1 target context",
)

require_parquet_columns(
    FINAL_421_QA_PATH,
    ["check", "status", "detail"],
    "4.2.1 final QA",
)

require_parquet_columns(
    PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH,
    ["feature_name"],
    "Pre-CP recursive feature contract",
)


forecast_feature_catalog_421_df = pd.read_parquet(
    FORECAST_FEATURE_CATALOG_PATH
)

forecast_target_context_df = pd.read_parquet(
    FORECAST_TARGET_CONTEXT_PATH
)

final_feature_contract_qa_421_df = pd.read_parquet(
    FINAL_421_QA_PATH
)

pre_cp_counterfactual_calendar_df = pd.read_parquet(
    PRE_CP_COUNTERFACTUAL_CALENDAR_PATH
)

pre_cp_recursive_feature_contract_421_df = pd.read_parquet(
    PRE_CP_RECURSIVE_FEATURE_CONTRACT_PATH
)


# Normalize the two calendar dates that 4.2.2 actually uses.
forecast_target_context_df["origin_date"] = pd.to_datetime(
    forecast_target_context_df["origin_date"]
)

forecast_target_context_df["target_date"] = pd.to_datetime(
    forecast_target_context_df["target_date"]
)


# The orientation audit confirmed this exact key has zero duplicates.
validate_unique_key(
    forecast_target_context_df,
    TARGET_CONTEXT_KEYS,
    "4.2.1 forecast-target context",
)

The chart below provides a quick visual check of the inherited forecasting populations. The three tracks intentionally cover different historical windows, so their physical-origin counts should differ. The important requirement is that the populations remain intact rather than being silently reduced to a smaller comparison cohort.

In [5]:
# 4.2.2.1C — Visualize inherited physical-origin populations

origin_population_plot_df = (
    origin_surface_summary_df[
        [
            "modeling_track",
            "physical_origins",
        ]
    ]
    .copy()
)

origin_population_plot_df["track_label"] = (
    origin_population_plot_df["modeling_track"]
    .map(TRACK_LABELS)
)

origin_population_plot_df["physical_origins_thousands"] = (
    origin_population_plot_df["physical_origins"]
    / 1_000
).round(3)

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=origin_population_plot_df["physical_origins_thousands"],
        y=origin_population_plot_df["track_label"],
        orientation="h",
        marker_color=BRAND_COLORS["dark_teal"],
        text=origin_population_plot_df["physical_origins_thousands"],
        texttemplate="%{text:.3f}k",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Physical origins: %{x:.3f}k"
            "<extra></extra>"
        ),
    )
)

apply_chart_layout(
    fig,
    title="Inherited Forecast-Origin Population by Modeling Track",
    x_title="Physical origins (thousands)",
    y_title=None,
    height=420,
)

fig.update_yaxes(
    autorange="reversed"
)

fig.show()

Findings\. The inherited physical\-origin populations preserve the intended difference in historical scope: Full\-History Regime\-Aware has the largest origin surface, Pre\-CP Reference is next, and Post\-CP Primary is smallest because it begins only on January 5, 2025\. Across the three tracks, the later contract QA confirms 3,056,852 unique Modeling Track × Taxi Zone × origin rows, so the context build is operating on the native track populations rather than reducing them to a smaller cross\-regime comparison cohort\.

**Load canonical mobility and align it to forecast origins.** Load the canonical mobility panel and align each physical forecast origin with the observed transportation state at that exact sequence position.

No filling or interpolation is performed here. If a mobility measure is unavailable at an origin, it stays missing. The date, Daypart, and temporal bucket carried forward from 4.2.1 are independently checked against the canonical mobility calendar so a sequence-alignment error cannot silently create a future-information leak.

The congestion-pricing sequence boundary is also derived from the calendar itself rather than trusted only as a hard-coded row number.

In [6]:
# 4.2.2.1D — Load canonical mobility and align it to forecast origins

MOBILITY_REQUIRED_COLUMNS = [
    "taxi_zone_id",
    "zone",
    "borough",
    "canonical_location_id",
    "date",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
    *MOBILITY_CONTEXT_METRICS,
]

require_parquet_columns(
    MOBILITY_PANEL_PATH,
    MOBILITY_REQUIRED_COLUMNS,
    "Analysis-ready mobility panel",
)

mobility_context_source_df = pd.read_parquet(
    MOBILITY_PANEL_PATH,
    columns=MOBILITY_REQUIRED_COLUMNS,
)

mobility_context_source_df["date"] = pd.to_datetime(
    mobility_context_source_df["date"]
)

validate_unique_key(
    mobility_context_source_df,
    CANONICAL_MOBILITY_KEYS,
    "Analysis-ready mobility panel",
)


# ---------------------------------------------------------------------
# Recover the canonical daypart sequence
# ---------------------------------------------------------------------

canonical_calendar_df = (
    mobility_context_source_df[
        [
            "observation_sequence_id",
            "date",
            "daypart",
            "daypart_order",
            "temporal_bucket",
            "temporal_bucket_order",
            "pre_post_cp",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "observation_sequence_id"
    )
    .reset_index(drop=True)
)

validate_unique_key(
    canonical_calendar_df,
    ["observation_sequence_id"],
    "Canonical daypart calendar",
)


# Derive the first post-CP sequence position directly from the calendar.
cp_start_calendar = (
    canonical_calendar_df.loc[
        canonical_calendar_df["date"].ge(
            CONGESTION_PRICING_START_DATE
        )
    ]
    .sort_values(
        "observation_sequence_id"
    )
    .iloc[0]
)

CP_FIRST_OBSERVATION_SEQUENCE_ID = int(
    cp_start_calendar[
        "observation_sequence_id"
    ]
)

PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID = (
    CP_FIRST_OBSERVATION_SEQUENCE_ID
    - 1
)

if pd.Timestamp(
    cp_start_calendar["date"]
) != CONGESTION_PRICING_START_DATE:
    raise ValueError(
        "The canonical sequence does not place the first post-CP "
        "observation on 2025-01-05."
    )

if str(
    cp_start_calendar["daypart"]
) != "overnight":
    raise ValueError(
        "The first post-CP canonical position is not Overnight."
    )


# ---------------------------------------------------------------------
# Attach observed at-origin mobility state
# ---------------------------------------------------------------------

mobility_origin_source_df = (
    mobility_context_source_df[
        [
            *CANONICAL_MOBILITY_KEYS,
            "date",
            "daypart",
            "daypart_order",
            "temporal_bucket",
            "temporal_bucket_order",
            "pre_post_cp",
            *MOBILITY_CONTEXT_METRICS,
        ]
    ]
    .copy()
)

mobility_origin_source_df[
    "_mobility_source_row_present"
] = True

origin_mobility_context_df = (
    forecast_physical_origin_df
    .merge(
        mobility_origin_source_df,
        on=CANONICAL_MOBILITY_KEYS,
        how="left",
        validate="many_to_one",
        suffixes=("", "_canonical"),
    )
)

origin_mobility_context_df[
    "canonical_origin_date_matches"
] = (
    pd.to_datetime(
        origin_mobility_context_df["origin_date"]
    )
    .eq(
        pd.to_datetime(
            origin_mobility_context_df["date"]
        )
    )
)

origin_mobility_context_df[
    "canonical_origin_daypart_matches"
] = (
    origin_mobility_context_df[
        "origin_daypart"
    ]
    .astype(str)
    .eq(
        origin_mobility_context_df[
            "daypart"
        ].astype(str)
    )
)

origin_mobility_context_df[
    "canonical_origin_bucket_matches"
] = (
    origin_mobility_context_df[
        "origin_temporal_bucket"
    ]
    .astype(str)
    .eq(
        origin_mobility_context_df[
            "temporal_bucket"
        ].astype(str)
    )
)

# ---------------------------------------------------------------------
# Confirm that every inherited physical origin reached the canonical mobility
# state expected for its Taxi Zone and sequence position. Detailed contract
# assertions follow in 4.2.2.1F; this output shows the result of the alignment
# operation itself.
# ---------------------------------------------------------------------

mobility_alignment_summary_records = []

for track in MODELING_TRACKS:
    track_df = (
        origin_mobility_context_df.loc[
            origin_mobility_context_df[
                "modeling_track"
            ].eq(track)
        ]
    )

    calendar_match = (
        track_df["canonical_origin_date_matches"]
        & track_df["canonical_origin_daypart_matches"]
        & track_df["canonical_origin_bucket_matches"]
    )

    mobility_alignment_summary_records.append({
        "modeling_track": TRACK_LABELS[track],
        "physical_origins": len(track_df),
        "taxi_zones": track_df["taxi_zone_id"].nunique(),
        "first_sequence": int(
            track_df["observation_sequence_id"].min()
        ),
        "last_sequence": int(
            track_df["observation_sequence_id"].max()
        ),
        "first_origin_date": track_df["origin_date"].min(),
        "last_origin_date": track_df["origin_date"].max(),
        "mobility_source_match": round(
            float(
                track_df[
                    "_mobility_source_row_present"
                ]
                .fillna(False)
                .mean()
            ),
            3,
        ),
        "calendar_alignment": round(
            float(
                calendar_match.mean()
            ),
            3,
        ),
    })


mobility_alignment_summary_df = pd.DataFrame(
    mobility_alignment_summary_records
)

display(
    mobility_alignment_summary_df
)

,modeling_track,physical_origins,taxi_zones,first_sequence,last_sequence,first_origin_date,last_origin_date,mobility_source_match,calendar_alignment
0,Post-CP Primary,579278,257,3676,5929,2025-01-05,2026-03-31,1.000,1.000
1,Full-History Regime-Aware,1529682,258,1,5929,2023-01-01,2026-03-31,1.000,1.000
2,Pre-CP Reference,948150,258,1,3675,2023-01-01,2025-01-04,1.000,1.000


Findings\. All three modeling tracks align cleanly to the canonical mobility panel: both mobility\-source matching and calendar alignment are 1\.000 for every track\. The inherited windows also reproduce the intended forecasting boundaries exactly—Post\-CP Primary begins at sequence 3,676 on January 5, 2025; Pre\-CP Reference ends at sequence 3,675 on January 4, 2025; and Full\-History Regime\-Aware spans the complete sequence from 1 through 5,929\. The aligned populations contain 579,278 Post\-CP, 1,529,682 Full\-History, and 947,892 Pre\-CP physical origins, with 257–258 Taxi Zones represented depending on track\.

**Load static Taxi Zone context and transportation connectivity.** Load the authoritative Taxi Zone context and transportation-connectivity sources. The static reference is validated now and will be attached after the dynamic transportation-connected context is constructed in Section 4.2.2.3. The connectivity table is retained using its actual upstream structure so the spatial features can be built from the authoritative network rather than from an assumed source-zone/neighbor-zone schema.

In [7]:
# 4.2.2.1E — Load static Taxi Zone context and transportation connectivity

# ---------------------------------------------------------------------
# 1. Static Taxi Zone context
#
# Upstream 1.3.1 names the raw identifier `location_id`. Rename it once at
# the notebook boundary so all downstream joins use the mobility-panel name
# `taxi_zone_id`.
# ---------------------------------------------------------------------

require_parquet_columns(
    TAXI_ZONE_SPATIAL_CONTEXT_PATH,
    STATIC_ZONE_SOURCE_COLUMNS,
    "Taxi Zone spatial context",
)

taxi_zone_context_df = (
    pd.read_parquet(
        TAXI_ZONE_SPATIAL_CONTEXT_PATH,
        columns=STATIC_ZONE_SOURCE_COLUMNS,
    )
    .rename(
        columns={
            "location_id": "taxi_zone_id",
        }
    )
)

validate_unique_key(
    taxi_zone_context_df,
    ["taxi_zone_id"],
    "Taxi Zone spatial context",
)


# ---------------------------------------------------------------------
# Verify that the ID normalization agrees with the mobility panel.
#
# We do not require the static reference and forecasting population to have
# identical row counts. Instead, every Taxi Zone used for forecasting must
# resolve to one static reference row, and its canonical_location_id must agree.
# ---------------------------------------------------------------------

mobility_zone_lookup_df = (
    mobility_context_source_df[
        [
            "taxi_zone_id",
            "canonical_location_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)

validate_unique_key(
    mobility_zone_lookup_df,
    ["taxi_zone_id"],
    "Mobility-panel Taxi Zone lookup",
)

static_crosswalk_qa_df = (
    mobility_zone_lookup_df
    .merge(
        taxi_zone_context_df[
            [
                "taxi_zone_id",
                "canonical_location_id",
            ]
        ].rename(
            columns={
                "canonical_location_id":
                    "static_canonical_location_id",
            }
        ),
        on="taxi_zone_id",
        how="left",
        validate="one_to_one",
        indicator="_static_join",
    )
)

static_crosswalk_qa_df[
    "canonical_location_id_matches"
] = (
    static_crosswalk_qa_df[
        "canonical_location_id"
    ].eq(
        static_crosswalk_qa_df[
            "static_canonical_location_id"
        ]
    )
)


# ---------------------------------------------------------------------
# 2. Transportation-aware Taxi Zone connectivity
#
# The upstream schema is now known exactly. Keep its directional relationship
# representation intact; 4.2.2.3 will decide how those connections should be
# summarized into forecasting features.
# ---------------------------------------------------------------------

CONNECTIVITY_REQUIRED_COLUMNS = [
    "location_id",
    "canonical_location_id",
    "zone",
    "borough",
    "connected_location_id",
    "connected_canonical_location_id",
    "connected_zone",
    "connected_borough",
    "connectivity_type",
]

require_parquet_columns(
    TAXI_ZONE_CONNECTIVITY_PATH,
    CONNECTIVITY_REQUIRED_COLUMNS,
    "Taxi Zone connectivity",
)

taxi_zone_connectivity_df = pd.read_parquet(
    TAXI_ZONE_CONNECTIVITY_PATH,
    columns=CONNECTIVITY_REQUIRED_COLUMNS,
)

if taxi_zone_connectivity_df.empty:
    raise ValueError(
        "Taxi Zone connectivity table is empty."
    )


# ---------------------------------------------------------------------
# Relationship-grain diagnostics
#
# Do not assume yet whether connectivity_type is required in the edge key.
# These diagnostics establish that empirically before 4.2.2.3 constructs
# connected-zone averages.
# ---------------------------------------------------------------------

connectivity_pair_duplicates = int(
    taxi_zone_connectivity_df.duplicated(
        [
            "canonical_location_id",
            "connected_canonical_location_id",
        ]
    ).sum()
)

connectivity_typed_pair_duplicates = int(
    taxi_zone_connectivity_df.duplicated(
        [
            "canonical_location_id",
            "connected_canonical_location_id",
            "connectivity_type",
        ]
    ).sum()
)

connectivity_contract_df = pd.DataFrame(
    [
        {
            "diagnostic": "Relationship rows",
            "value": len(taxi_zone_connectivity_df),
        },
        {
            "diagnostic": "Unique source canonical zones",
            "value": taxi_zone_connectivity_df[
                "canonical_location_id"
            ].nunique(),
        },
        {
            "diagnostic": "Unique connected canonical zones",
            "value": taxi_zone_connectivity_df[
                "connected_canonical_location_id"
            ].nunique(),
        },
        {
            "diagnostic": "Duplicate source-connected pairs",
            "value": connectivity_pair_duplicates,
        },
        {
            "diagnostic": "Duplicate source-connected-type rows",
            "value": connectivity_typed_pair_duplicates,
        },
    ]
)

display(connectivity_contract_df)


connectivity_type_counts_df = (
    taxi_zone_connectivity_df[
        "connectivity_type"
    ]
    .value_counts(dropna=False)
    .rename_axis("connectivity_type")
    .reset_index(name="relationship_rows")
)

display(connectivity_type_counts_df)

,diagnostic,value
0,Relationship rows,876
1,Unique source canonical zones,259
2,Unique connected canonical zones,259
3,Duplicate source-connected pairs,0
4,Duplicate source-connected-type rows,0


,connectivity_type,relationship_rows
0,polygon_touching,686
1,transportation_corridor,104
2,zero_distance_geometry_fallback,86


Findings\. The connectivity source contains 876 source–connected Taxi Zone relationships spanning 259 canonical zones on each side of the relationship, with no duplicate source–connected pairs even before connectivity type is considered\. Most links are polygon\-touching relationships \(686\), while 104 are explicit transportation\-corridor connections and 86 use the zero\-distance geometry fallback\. This gives the spatial feature step a clean, non\-duplicated relationship table that captures both ordinary geographic adjacency and additional transportation\-aware links\.

**Validate the complete forecasting-context input contract.** Run one consolidated QA check across the complete input handoff. Passing this block means the new notebook is using the intended forecasting populations, every origin resolves to the canonical mobility panel, track boundaries remain intact, and the static/spatial sources cover the required forecasting geography.

These are validation checks only. No feature-selection decision is made here.

In [8]:
# 4.2.2.1F — Validate the complete forecasting-context input contract

input_qa_rows = []


# ---------------------------------------------------------------------
# 1. Completed 4.2.1 contract
# ---------------------------------------------------------------------

input_qa_rows.append(
    qa_row(
        "All upstream 4.2.1 final QA checks still pass",
        final_feature_contract_qa_421_df[
            "status"
        ].eq("PASS").all(),
        (
            f"{int(final_feature_contract_qa_421_df['status'].eq('PASS').sum()):,}/"
            f"{len(final_feature_contract_qa_421_df):,} checks pass"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Physical forecast-origin registry is unique",
        not forecast_physical_origin_df.duplicated(
            PHYSICAL_ORIGIN_KEYS
        ).any(),
        (
            f"{len(forecast_physical_origin_df):,} unique "
            "track × Taxi Zone × origin rows"
        ),
    )
)


# ---------------------------------------------------------------------
# 2. 4.2.1 target-calendar contract
# ---------------------------------------------------------------------

input_qa_rows.append(
    qa_row(
        "Target-context key is unique",
        not forecast_target_context_df.duplicated(
            TARGET_CONTEXT_KEYS
        ).any(),
        (
            f"{len(forecast_target_context_df):,} "
            "track × origin × horizon rows"
        ),
    )
)

actual_target_tracks = set(
    forecast_target_context_df[
        "modeling_track"
    ].dropna().unique()
)

input_qa_rows.append(
    qa_row(
        "Target context contains exactly the three modeling tracks",
        actual_target_tracks == set(MODELING_TRACKS),
        f"tracks = {sorted(actual_target_tracks)}",
    )
)

actual_horizons = set(
    pd.to_numeric(
        forecast_target_context_df[
            "horizon"
        ],
        errors="raise",
    )
    .astype(int)
    .unique()
)

input_qa_rows.append(
    qa_row(
        "Target context contains exactly h=1, h=2, and h=5",
        actual_horizons == {1, 2, 5},
        f"horizons = {sorted(actual_horizons)}",
    )
)


# ---------------------------------------------------------------------
# 3. Canonical mobility alignment
# ---------------------------------------------------------------------

missing_origin_source_rows = int(
    origin_mobility_context_df[
        "_mobility_source_row_present"
    ].isna().sum()
)

input_qa_rows.append(
    qa_row(
        "Every forecast origin resolves to canonical mobility",
        missing_origin_source_rows == 0,
        f"unmatched origins = {missing_origin_source_rows:,}",
    )
)

input_qa_rows.append(
    qa_row(
        "Forecast-origin dates match the canonical calendar",
        origin_mobility_context_df[
            "canonical_origin_date_matches"
        ].all(),
        (
            "mismatches = "
            f"{int((~origin_mobility_context_df['canonical_origin_date_matches']).sum()):,}"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Forecast-origin Dayparts match the canonical calendar",
        origin_mobility_context_df[
            "canonical_origin_daypart_matches"
        ].all(),
        (
            "mismatches = "
            f"{int((~origin_mobility_context_df['canonical_origin_daypart_matches']).sum()):,}"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Forecast-origin temporal buckets match the canonical calendar",
        origin_mobility_context_df[
            "canonical_origin_bucket_matches"
        ].all(),
        (
            "mismatches = "
            f"{int((~origin_mobility_context_df['canonical_origin_bucket_matches']).sum()):,}"
        ),
    )
)


# ---------------------------------------------------------------------
# 4. Exact congestion-pricing sequence boundary
#
# The canonical calendar must independently reproduce the already-established
# transition:
#   3675 = Jan 4 Evening, Pre-CP
#   3676 = Jan 5 Overnight, Post-CP
# ---------------------------------------------------------------------

freeze_row = canonical_calendar_df.loc[
    canonical_calendar_df[
        "observation_sequence_id"
    ].eq(
        EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID
    )
]

cp_first_row = canonical_calendar_df.loc[
    canonical_calendar_df[
        "observation_sequence_id"
    ].eq(
        EXPECTED_CP_FIRST_SEQUENCE_ID
    )
]

freeze_boundary_ok = (
    len(freeze_row) == 1
    and pd.Timestamp(
        freeze_row.iloc[0]["date"]
    ) == PRE_CP_FREEZE_DATE
    and str(
        freeze_row.iloc[0]["daypart"]
    ) == "evening"
    and str(
        freeze_row.iloc[0]["pre_post_cp"]
    ) == "pre_cp"
)

cp_start_boundary_ok = (
    len(cp_first_row) == 1
    and pd.Timestamp(
        cp_first_row.iloc[0]["date"]
    ) == CONGESTION_PRICING_START_DATE
    and str(
        cp_first_row.iloc[0]["daypart"]
    ) == "overnight"
    and str(
        cp_first_row.iloc[0]["pre_post_cp"]
    ) == "post_cp"
)

input_qa_rows.append(
    qa_row(
        "Canonical sequence reproduces the final Pre-CP origin",
        freeze_boundary_ok,
        (
            f"expected sequence {EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID:,} = "
            "2025-01-04 Evening, Pre-CP"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Canonical sequence reproduces the first Post-CP origin",
        cp_start_boundary_ok,
        (
            f"expected sequence {EXPECTED_CP_FIRST_SEQUENCE_ID:,} = "
            "2025-01-05 Overnight, Post-CP"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Derived CP sequence boundary matches the established contract",
        (
            CP_FIRST_OBSERVATION_SEQUENCE_ID
            == EXPECTED_CP_FIRST_SEQUENCE_ID
            and PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID
            == EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID
        ),
        (
            f"derived freeze/start = "
            f"{PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID:,}/"
            f"{CP_FIRST_OBSERVATION_SEQUENCE_ID:,}"
        ),
    )
)


# ---------------------------------------------------------------------
# 5. Modeling-track boundaries
# ---------------------------------------------------------------------

post_origins = (
    forecast_physical_origin_df.loc[
        forecast_physical_origin_df[
            "modeling_track"
        ].eq(
            "post_cp_primary"
        )
    ]
)

pre_origins = (
    forecast_physical_origin_df.loc[
        forecast_physical_origin_df[
            "modeling_track"
        ].eq(
            "pre_cp_reference"
        )
    ]
)

input_qa_rows.append(
    qa_row(
        "Post-CP Primary origins never reach before sequence 3676",
        (
            post_origins[
                "observation_sequence_id"
            ]
            .ge(
                EXPECTED_CP_FIRST_SEQUENCE_ID
            )
            .all()
            and post_origins[
                "origin_date"
            ]
            .ge(
                CONGESTION_PRICING_START_DATE
            )
            .all()
        ),
        (
            "first sequence/date = "
            f"{post_origins['observation_sequence_id'].min():,} / "
            f"{post_origins['origin_date'].min():%Y-%m-%d}"
        ),
    )
)

input_qa_rows.append(
    qa_row(
        "Pre-CP Reference observed origins never extend beyond sequence 3675",
        (
            pre_origins[
                "observation_sequence_id"
            ]
            .le(
                EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID
            )
            .all()
            and pre_origins[
                "origin_date"
            ]
            .le(
                PRE_CP_FREEZE_DATE
            )
            .all()
        ),
        (
            "last sequence/date = "
            f"{pre_origins['observation_sequence_id'].max():,} / "
            f"{pre_origins['origin_date'].max():%Y-%m-%d}"
        ),
    )
)


# ---------------------------------------------------------------------
# 6. Static Taxi Zone crosswalk
# ---------------------------------------------------------------------

unmatched_static_zones = int(
    static_crosswalk_qa_df[
        "_static_join"
    ].ne("both").sum()
)

canonical_id_mismatches = int(
    (
        static_crosswalk_qa_df[
            "_static_join"
        ].eq("both")
        &
        ~static_crosswalk_qa_df[
            "canonical_location_id_matches"
        ]
    ).sum()
)

input_qa_rows.append(
    qa_row(
        "Static Taxi Zone context covers every mobility-panel zone",
        unmatched_static_zones == 0,
        f"unmatched Taxi Zones = {unmatched_static_zones:,}",
    )
)

input_qa_rows.append(
    qa_row(
        "Static and mobility canonical-location IDs agree",
        canonical_id_mismatches == 0,
        f"canonical-ID mismatches = {canonical_id_mismatches:,}",
    )
)


# ---------------------------------------------------------------------
# 7. Connectivity handoff
#
# Availability is asserted here. Exact edge-grain interpretation is intentionally
# left to the observed diagnostics from 4.2.2.1E rather than guessed.
# ---------------------------------------------------------------------

input_qa_rows.append(
    qa_row(
        "Transportation-connectivity source is available",
        len(taxi_zone_connectivity_df) > 0,
        (
            f"{len(taxi_zone_connectivity_df):,} relationship rows; "
            f"{taxi_zone_connectivity_df['canonical_location_id'].nunique():,} "
            "source canonical zones"
        ),
    )
)


# ---------------------------------------------------------------------
# Consolidated reader-facing QA
# ---------------------------------------------------------------------

context_input_qa_df = pd.DataFrame(
    input_qa_rows
)

display(
    context_input_qa_df
)

failed_input_checks = (
    context_input_qa_df.loc[
        context_input_qa_df[
            "status"
        ].ne("PASS")
    ]
)

if not failed_input_checks.empty:
    raise AssertionError(
        "4.2.2 input QA failed: "
        + "; ".join(
            failed_input_checks[
                "check"
            ]
        )
    )

,check,status,detail
0,All upstream 4.2.1 final QA checks still pass,PASS,24/24 checks pass
1,Physical forecast-origin registry is unique,PASS,"3,057,110 unique track × Taxi Zone × origin rows"
2,Target-context key is unique,PASS,"35,496 track × origin × horizon rows"
3,Target context contains exactly the three modeling tracks,PASS,"tracks = ['full_history_regime_aware', 'post_cp_primary', 'pre_cp_reference']"
4,"Target context contains exactly h=1, h=2, and h=5",PASS,"horizons = [1, 2, 5]"
5,Every forecast origin resolves to canonical mobility,PASS,unmatched origins = 0
6,Forecast-origin dates match the canonical calendar,PASS,mismatches = 0
7,Forecast-origin Dayparts match the canonical calendar,PASS,mismatches = 0
8,Forecast-origin temporal buckets match the canonical calendar,PASS,mismatches = 0
9,Canonical sequence reproduces the final Pre-CP origin,PASS,"expected sequence 3,675 = 2025-01-04 Evening, Pre-CP"


Findings\. All 17 input\-contract checks pass\. Every one of the 3,056,852 physical forecast origins resolves to the canonical mobility panel with zero date, Daypart, or temporal\-bucket mismatches; the 35,496 target\-context rows are unique at Modeling Track × origin × horizon and contain exactly h=1, h=2, and h=5\. The congestion\-pricing boundary is reproduced exactly at sequence 3,675 → 3,676, the Post\-CP and Pre\-CP track limits are correct, the static Taxi Zone crosswalk has zero unmatched zones or canonical\-ID mismatches, and the 876\-row connectivity source is available for the next spatial step\.

# 4.2.2.2 Construct Multimodal Historical Features

A forecast for one mobility measure may benefit from seeing the recent state of the other transportation measures in the same Taxi Zone.

**Build leakage-safe multimodal origin states.** Start with a deliberately simple representation of the broader transportation system. For each mobility measure, construct:

- its value at the forecast origin; and
- its change from the **immediately preceding canonical Daypart**.

For the five forecast targets, this temporarily overlaps with the target's own recent-history features already built in 4.2.1. That overlap is intentional: 4.2.2 first builds one **target-neutral mobility state** that can be reused for every forecast target. During final target-specific assembly, the forecast target's own level and one-step change will be removed, leaving 4.2.1 responsible for that series' history and 4.2.2 responsible for cross-mode context.

This gives downstream Tree and neural models direct information about the broader transportation system without hard-coding a large collection of ratios and interaction formulas. Missing count-like observations are treated as zero activity, consistent with the project's established preprocessing rules; missing speed and duration remain missing.

The previous-step change is only valid when the preceding canonical sequence position exists and is permitted by the modeling track. Post-CP Primary cannot form its first January 5 change from the final Pre-CP state, Full-History Regime-Aware remains continuous across that boundary, and Pre-CP Reference remains limited to its observed history through January 4.

The candidate state is built once per **Modeling Track × Taxi Zone × Origin observation_sequence_id** and then reused across forecast targets.

In [9]:
# 4.2.2.2A — Build leakage-safe multimodal origin states
#              with explicit missing-data semantics


# ---------------------------------------------------------------------
# Metric semantics
# ---------------------------------------------------------------------
# Count/activity measures use the established interpretation that an absent
# observation represents zero observed activity / structural non-participation.
# That semantic zero is applied BEFORE origin levels and Δ1 are constructed.
COUNT_CONTEXT_METRICS = [
    "taxi_trip_count",
    "fhvhv_trip_count",
    "subway_ridership",
    "subway_transfers",
    "bus_trip_count",
]

# Performance measures behave differently. A missing speed or duration does
# not mean zero performance, so missing values remain missing here.
#
# Any later model-specific numerical imputation must be fitted only on that
# model's legally available training data; 4.2.2 deliberately does not fit
# such values across the full historical panel.
PERFORMANCE_CONTEXT_METRICS = [
    "taxi_avg_trip_speed",
    "taxi_avg_trip_duration",
    "fhvhv_avg_trip_speed",
    "fhvhv_avg_trip_duration",
    "avg_bus_speed",
]

assert not (
    set(COUNT_CONTEXT_METRICS)
    & set(PERFORMANCE_CONTEXT_METRICS)
), "Count/activity and performance metric groups must not overlap."

assert (
    set(COUNT_CONTEXT_METRICS)
    | set(PERFORMANCE_CONTEXT_METRICS)
) == set(MOBILITY_CONTEXT_METRICS), (
    "Every multimodal context metric must have exactly one missingness policy."
)


# ---------------------------------------------------------------------
# Stable support/provenance column names
# ---------------------------------------------------------------------
def multimodal_origin_missing_flag_column(metric: str) -> str:
    """
    Flag whether the raw at-origin source value was missing.

    For count/activity measures:
        True means the feature value was semantically replaced with zero.

    For performance measures:
        True means the feature value remains missing.
    """
    return f"mm_{metric}_origin_missing_flag"


def multimodal_delta_missing_input_flag_column(metric: str) -> str:
    """
    Flag whether either raw endpoint needed by Δ1 was missing.

    The flag is null where the preceding canonical step itself is not legal
    under the modeling-track boundary, because Δ1 is undefined there for a
    timing reason rather than a source-missingness reason.
    """
    return f"mm_{metric}_seq_delta_1_missing_input_flag"


# ---------------------------------------------------------------------
# Build one multimodal context row per physical forecast origin
# ---------------------------------------------------------------------
def build_multimodal_candidate_state() -> pd.DataFrame:
    """
    Construct multimodal levels and exact one-step changes at forecast origins.

    Grain
    -----
    Modeling Track × Taxi Zone × origin observation_sequence_id.

    Timing
    ------
    Origin-level values are available when the forecast is issued.

    Δ1 compares the origin with exactly observation_sequence_id - 1.
    Missing canonical sequence positions are never jumped across.

    Missing-value semantics
    -----------------------
    Count/activity:
        Missing raw values -> semantic zero before level / Δ1 construction.

    Performance:
        Missing raw values remain missing. No retrospective/global median is
        fitted in this feature-construction notebook.

    Track boundaries
    ----------------
    Post-CP Primary begins fresh on the first Post-CP observation.
    Full-History Regime-Aware remains historically continuous.
    Pre-CP Reference remains inside its approved observed-history window.
    """

    # Current physical mobility state for every legal forecast origin.
    current = (
        origin_mobility_context_df[
            [
                "modeling_track",
                "taxi_zone_id",
                "observation_sequence_id",
                "origin_date",
                "origin_daypart",
                "origin_daypart_order",
                "origin_temporal_bucket",
                "origin_temporal_bucket_order",
                *MOBILITY_CONTEXT_METRICS,
            ]
        ]
        .copy()
    )

    # Build a source table whose sequence ID is shifted forward by one. This
    # lets origin t join directly to the same Taxi Zone at canonical step t-1.
    previous = (
        mobility_context_source_df[
            [
                "taxi_zone_id",
                "observation_sequence_id",
                *MOBILITY_CONTEXT_METRICS,
            ]
        ]
        .copy()
    )

    previous[
        "_previous_observation_sequence_id"
    ] = previous["observation_sequence_id"]

    previous["observation_sequence_id"] = (
        previous["observation_sequence_id"] + 1
    )

    previous = previous.rename(
        columns={
            metric: f"_previous_{metric}"
            for metric in MOBILITY_CONTEXT_METRICS
        }
    )

    result = current.merge(
        previous,
        on=[
            "taxi_zone_id",
            "observation_sequence_id",
        ],
        how="left",
        validate="many_to_one",
    )

    # -----------------------------------------------------------------
    # Confirm that the joined row is the EXACT prior canonical step.
    # -----------------------------------------------------------------
    exact_previous = (
        result["_previous_observation_sequence_id"]
        .eq(
            result["observation_sequence_id"] - 1
        )
        .fillna(False)
    )

    legal_previous = exact_previous.copy()


    # -----------------------------------------------------------------
    # Post-CP Primary may not borrow the final Pre-CP mobility state.
    # -----------------------------------------------------------------
    post_mask = result["modeling_track"].eq(
        "post_cp_primary"
    )

    legal_previous &= ~(
        post_mask
        & result["_previous_observation_sequence_id"].lt(
            CP_FIRST_OBSERVATION_SEQUENCE_ID
        )
    )


    # -----------------------------------------------------------------
    # Pre-CP Reference may not cross its frozen observed-history boundary.
    # -----------------------------------------------------------------
    pre_mask = result["modeling_track"].eq(
        "pre_cp_reference"
    )

    legal_previous &= ~(
        pre_mask
        & (
            result["observation_sequence_id"].gt(
                PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID
            )
            | result["_previous_observation_sequence_id"].gt(
                PRE_CP_FREEZE_OBSERVATION_SEQUENCE_ID
            )
        )
    )

    result[
        "_previous_step_available_under_track"
    ] = legal_previous


    # -----------------------------------------------------------------
    # Construct the compact candidate library.
    #
    # Each metric contributes:
    #   1. its current state at the forecast origin;
    #   2. its exact one-canonical-step change.
    #
    # Support flags preserve whether the raw mobility input was absent.
    # They are provenance/support metadata here, not automatically promoted
    # to forecasting predictors.
    # -----------------------------------------------------------------
    feature_columns = []
    support_columns = []

    for metric in MOBILITY_CONTEXT_METRICS:
        level_col = multimodal_level_column(metric)
        delta_col = multimodal_delta_column(metric)

        origin_flag_col = (
            multimodal_origin_missing_flag_column(metric)
        )
        delta_flag_col = (
            multimodal_delta_missing_input_flag_column(metric)
        )

        previous_col = f"_previous_{metric}"

        origin_missing = result[metric].isna()
        delta_missing_input = (
            origin_missing
            | result[previous_col].isna()
        )

        # Preserve raw-source missingness explicitly.
        result[origin_flag_col] = (
            origin_missing.astype("boolean")
        )

        # Null means Δ1 was illegal because no legal prior canonical step
        # existed under the track—not that a source value was missing.
        result[delta_flag_col] = (
            delta_missing_input
            .where(
                legal_previous,
                pd.NA,
            )
            .astype("boolean")
        )

        if metric in COUNT_CONTEXT_METRICS:
            # Missing count/activity means zero activity. Apply this BEFORE
            # computing Δ1 so transitions into or out of structural zero are
            # represented as actual mobility changes.
            current_clean = result[metric].fillna(0.0)
            previous_clean = result[previous_col].fillna(0.0)

            result[level_col] = current_clean

            result[delta_col] = (
                current_clean - previous_clean
            ).where(
                legal_previous
            )

        else:
            # Missing performance remains missing. In particular, do not turn
            # absent service into zero speed/duration and do not fit a global
            # median using future observations.
            result[level_col] = result[metric]

            result[delta_col] = (
                result[metric]
                - result[previous_col]
            ).where(
                legal_previous
            )

        feature_columns.extend(
            [
                level_col,
                delta_col,
            ]
        )

        support_columns.extend(
            [
                origin_flag_col,
                delta_flag_col,
            ]
        )


    # -----------------------------------------------------------------
    # Keep the origin contract, predictor candidates, and support metadata.
    # -----------------------------------------------------------------
    output = (
        result[
            [
                *PHYSICAL_ORIGIN_KEYS,
                "origin_date",
                "origin_daypart",
                "origin_daypart_order",
                "origin_temporal_bucket",
                "origin_temporal_bucket_order",
                "_previous_observation_sequence_id",
                "_previous_step_available_under_track",
                *feature_columns,
                *support_columns,
            ]
        ]
        .sort_values(
            [
                "modeling_track",
                "taxi_zone_id",
                "observation_sequence_id",
            ]
        )
        .reset_index(drop=True)
    )

    validate_unique_key(
        output,
        PHYSICAL_ORIGIN_KEYS,
        "Multimodal candidate origin state",
    )

    return output


# Force a fresh build while REBUILD_MULTIMODAL_CANDIDATE_STATE = True.
multimodal_candidate_state_df = load_or_build_parquet(
    MULTIMODAL_CANDIDATE_STATE_PATH,
    REBUILD_MULTIMODAL_CANDIDATE_STATE,
    build_multimodal_candidate_state,
)


# ---------------------------------------------------------------------
# Guard against accidentally reloading the older cached schema.
# ---------------------------------------------------------------------
required_support_columns = [
    multimodal_origin_missing_flag_column(metric)
    for metric in MOBILITY_CONTEXT_METRICS
] + [
    multimodal_delta_missing_input_flag_column(metric)
    for metric in MOBILITY_CONTEXT_METRICS
]

missing_support_columns = sorted(
    set(required_support_columns).difference(
        multimodal_candidate_state_df.columns
    )
)

if missing_support_columns:
    raise ValueError(
        "The multimodal candidate state does not contain the new "
        "missingness-support contract. Rebuild the cached artifact first. "
        f"Missing columns: {missing_support_columns}"
    )


# ---------------------------------------------------------------------
# Compact build validation
# ---------------------------------------------------------------------
# Count TRUE cells without allowing nullable Boolean values to distort totals.
def count_true_cells(
    frame: pd.DataFrame,
    columns: list[str],
) -> int:
    """Count True values across a set of nullable support-flag columns."""
    return int(
        frame[columns]
        .fillna(False)
        .astype(bool)
        .to_numpy()
        .sum()
    )


count_origin_flags = [
    multimodal_origin_missing_flag_column(metric)
    for metric in COUNT_CONTEXT_METRICS
]

count_delta_flags = [
    multimodal_delta_missing_input_flag_column(metric)
    for metric in COUNT_CONTEXT_METRICS
]

performance_origin_flags = [
    multimodal_origin_missing_flag_column(metric)
    for metric in PERFORMANCE_CONTEXT_METRICS
]

performance_delta_flags = [
    multimodal_delta_missing_input_flag_column(metric)
    for metric in PERFORMANCE_CONTEXT_METRICS
]

build_summary_records = []

for modeling_track, track_rows in multimodal_candidate_state_df.groupby(
    "modeling_track",
    observed=True,
    sort=False,
):
    legal_prior = (
        track_rows["_previous_step_available_under_track"]
        .fillna(False)
        .astype(bool)
    )

    build_summary_records.append(
        {
            "modeling_track": modeling_track,
            "origin_rows": int(len(track_rows)),
            "taxi_zones": int(
                track_rows["taxi_zone_id"].nunique()
            ),
            "first_origin_sequence_id": int(
                track_rows["observation_sequence_id"].min()
            ),
            "last_origin_sequence_id": int(
                track_rows["observation_sequence_id"].max()
            ),
            "legal_prior_step_share": round(
                float(legal_prior.mean()),
                3,
            ),
            "count_level_semantic_zero_fills": (
                count_true_cells(
                    track_rows,
                    count_origin_flags,
                )
            ),
            "count_delta_features_using_semantic_zero": (
                count_true_cells(
                    track_rows.loc[legal_prior],
                    count_delta_flags,
                )
            ),
            "performance_level_missing_cells": (
                count_true_cells(
                    track_rows,
                    performance_origin_flags,
                )
            ),
            "performance_delta_features_blocked_by_missing_input": (
                count_true_cells(
                    track_rows.loc[legal_prior],
                    performance_delta_flags,
                )
            ),
        }
    )

multimodal_candidate_build_summary_df = pd.DataFrame(
    build_summary_records
)

display(multimodal_candidate_build_summary_df)

,modeling_track,origin_rows,taxi_zones,first_origin_sequence_id,last_origin_sequence_id,legal_prior_step_share,count_level_semantic_zero_fills,count_delta_features_using_semantic_zero,performance_level_missing_cells,performance_delta_features_blocked_by_missing_input
0,full_history_regime_aware,1529682,258,1,5929,1.000,1309999,1325300,900751,1173386
1,post_cp_primary,579278,257,3676,5929,1.000,502017,506501,214201,280767
2,pre_cp_reference,948150,258,1,3675,1.000,800560,810635,679842,884499


Findings\. The multimodal origin state preserves the full native forecasting populations and the intended sequence boundaries for all three modeling tracks, with essentially complete access to a legal immediately preceding Daypart\. Applying the established missing\-data semantics materially changes the usable mobility state: across the five count/activity measures, 1,309,999 Full\-History, 502,017 Post\-CP, and 800,344 Pre\-CP feature cells are converted from missing observations to semantic zero activity, with a similar number of one\-step changes using at least one such zero endpoint\. Performance measures remain missing when speed or duration was not observed rather than being assigned artificial values; this leaves 900,751 Full\-History, 214,201 Post\-CP, and 679,746 Pre\-CP missing performance\-level cells, with missing inputs blocking even more performance Δ1 values\. The higher missingness in Pre\-CP performance context is preserved explicitly rather than hidden through retrospective imputation\.

**Register multimodal feature rules and availability.** Record how each multimodal candidate was constructed and where it may be used later. Count-like measures use the established zero-activity interpretation when observations are absent, while missing speed and duration remain missing rather than being assigned an artificial value.

The five forecast-target measures can later be rebuilt from recursively predicted no-CP mobility after January 4. The five auxiliary measures can still be used for ordinary forecasting, but their actual post-CP values cannot be used in the no-CP counterfactual unless those measures are separately forecast.

The catalog also marks same-target features for removal during final assembly because 4.2.1 already contains that target's own recent level and change.

In [10]:
# 4.2.2.2B — Register multimodal feature provenance,
#              missingness treatment, and runtime availability

multimodal_catalog_records = []

for metric in MOBILITY_CONTEXT_METRICS:
    is_forecast_target = metric in FORECAST_TARGETS
    is_count_metric = metric in COUNT_CONTEXT_METRICS

    metric_value_class = (
        "count/activity"
        if is_count_metric
        else "performance"
    )

    missing_value_policy = (
        "semantic zero before level / Δ1 construction"
        if is_count_metric
        else (
            "preserve missing; training-window preprocessing "
            "only if later required"
        )
    )

    if is_forecast_target:
        # After the Pre-CP freeze, these measures can be regenerated from the
        # model's recursively evolving hypothetical no-CP target state.
        counterfactual_class = "recursive_required"

        counterfactual_source = (
            "recompute from recursively predicted no-CP target state"
        )

        post_freeze_no_cp_rule = (
            "Recursive target state"
        )

    else:
        # These auxiliary measures are not themselves forecast. Their actual
        # post-CP observations therefore cannot enter the no-CP counterfactual.
        counterfactual_class = "ordinary_forecasting_only"

        counterfactual_source = (
            "unavailable after Pre-CP freeze unless separately forecast"
        )

        post_freeze_no_cp_rule = (
            "Ordinary forecasting only"
        )

    candidate_forms = [
        {
            "feature_form": "origin level",
            "feature_name": multimodal_level_column(metric),
            "support_flag": (
                multimodal_origin_missing_flag_column(metric)
            ),
        },
        {
            "feature_form": "exact prior-step change",
            "feature_name": multimodal_delta_column(metric),
            "support_flag": (
                multimodal_delta_missing_input_flag_column(metric)
            ),
        },
    ]

    for candidate in candidate_forms:
        multimodal_catalog_records.append(
            {
                # Preserve the existing catalog contract.
                "feature_name": candidate["feature_name"],
                "feature_family": "multimodal historical context",
                "source_metric": metric,
                "source_metric_label": METRIC_LABELS[metric],
                "feature_form": candidate["feature_form"],
                "ordinary_forecasting_class": (
                    "observed_at_or_before_origin"
                ),
                "pre_cp_post_freeze_class": counterfactual_class,
                "counterfactual_runtime_source": counterfactual_source,
                "drop_when_source_metric_equals_target_metric": (
                    is_forecast_target
                ),

                # New explicit missingness/provenance contract.
                "metric_value_class": metric_value_class,
                "missing_value_policy": missing_value_policy,
                "support_flag": candidate["support_flag"],
            }
        )


multimodal_candidate_catalog_df = pd.DataFrame(
    multimodal_catalog_records
)

assert len(multimodal_candidate_catalog_df) == (
    2 * len(MOBILITY_CONTEXT_METRICS)
), "Expected exactly two candidate predictor forms per source metric."

assert (
    multimodal_candidate_catalog_df["feature_name"].nunique()
    == len(multimodal_candidate_catalog_df)
), "Multimodal candidate feature names must be unique."


if WRITE_INTERMEDIATES:
    multimodal_candidate_catalog_df.to_parquet(
        MULTIMODAL_CANDIDATE_CATALOG_PATH,
        index=False,
    )


# ---------------------------------------------------------------------
# Reader-facing policy summary: one row per source mobility measure.
# ---------------------------------------------------------------------
multimodal_missingness_policy_df = (
    multimodal_candidate_catalog_df[
        [
            "source_metric",
            "source_metric_label",
            "metric_value_class",
            "missing_value_policy",
            "pre_cp_post_freeze_class",
        ]
    ]
    .drop_duplicates(
        subset=["source_metric"]
    )
    .copy()
)

multimodal_missingness_policy_df["role"] = (
    multimodal_missingness_policy_df[
        "source_metric"
    ].map(
        lambda metric: (
            "Forecast target"
            if metric in FORECAST_TARGETS
            else "Auxiliary context"
        )
    )
)

multimodal_missingness_policy_df[
    "post_freeze_no_cp_rule"
] = (
    multimodal_missingness_policy_df[
        "source_metric"
    ].map(
        lambda metric: (
            "Recursive target state"
            if metric in FORECAST_TARGETS
            else "Ordinary forecasting only"
        )
    )
)

multimodal_missingness_policy_df = (
    multimodal_missingness_policy_df[
        [
            "source_metric_label",
            "role",
            "metric_value_class",
            "missing_value_policy",
            "post_freeze_no_cp_rule",
        ]
    ]
    .rename(
        columns={
            "source_metric_label": "metric",
            "metric_value_class": "value_class",
            "missing_value_policy": "missingness_policy",
        }
    )
    .reset_index(drop=True)
)

display(multimodal_missingness_policy_df)

,metric,role,value_class,missingness_policy,post_freeze_no_cp_rule
0,Taxi trips,Forecast target,count/activity,semantic zero before level / Δ1 construction,Recursive target state
1,Taxi speed,Forecast target,performance,preserve missing; training-window preprocessing only if later required,Recursive target state
2,FHVHV trips,Forecast target,count/activity,semantic zero before level / Δ1 construction,Recursive target state
3,FHVHV speed,Forecast target,performance,preserve missing; training-window preprocessing only if later required,Recursive target state
4,Subway ridership,Forecast target,count/activity,semantic zero before level / Δ1 construction,Recursive target state
5,Taxi trip duration,Auxiliary context,performance,preserve missing; training-window preprocessing only if later required,Ordinary forecasting only
6,FHVHV trip duration,Auxiliary context,performance,preserve missing; training-window preprocessing only if later required,Ordinary forecasting only
7,Subway transfers,Auxiliary context,count/activity,semantic zero before level / Δ1 construction,Ordinary forecasting only
8,Bus trips,Auxiliary context,count/activity,semantic zero before level / Δ1 construction,Ordinary forecasting only
9,Bus speed,Auxiliary context,performance,preserve missing; training-window preprocessing only if later required,Ordinary forecasting only


Findings\. The multimodal feature contract  applies a consistent missingness rule across all 10 source measures\. Taxi trips, FHVHV trips, Subway ridership, Subway transfers, and Bus trips are treated as count/activity measures, so missing observations represent zero activity before levels and one\-step changes are calculated\. Taxi and FHVHV speed and duration, along with Bus speed, remain missing when performance was not observed; any later numerical preprocessing must therefore be learned within the legal model\-training window rather than across the full dataset\. The counterfactual boundary remains unchanged: features derived from the five forecast targets can be rebuilt from recursively predicted no\-CP mobility after January 4, while the five auxiliary measures remain available only for ordinary forecasting unless they are separately modeled\.

**Review multimodal candidate availability.** Measure how often each proposed multimodal feature is usable at the actual forecast origins after applying the project's missing-data rules.

For count-like measures, an absent observation can still produce a usable value because it represents zero activity. Speed and duration remain unavailable where they were not observed, and their one-step changes require valid observations at both the current and preceding Daypart.

The heatmap therefore shows **usable feature coverage**, while the hover details show how often the underlying source data was missing. This tells us whether the compact multimodal feature set is practical across the three modeling tracks before moving on to spatial context.

In [11]:
# 4.2.2.2C — Measure and visualize usable multimodal candidate coverage

coverage_records = []

coverage_track_order = [
    ("post_cp_primary", "Post"),
    ("full_history_regime_aware", "Full"),
    ("pre_cp_reference", "Pre"),
]


for metric in MOBILITY_CONTEXT_METRICS:
    level_col = multimodal_level_column(metric)
    delta_col = multimodal_delta_column(metric)

    origin_flag_col = multimodal_origin_missing_flag_column(metric)
    delta_flag_col = multimodal_delta_missing_input_flag_column(metric)

    is_count_metric = metric in COUNT_CONTEXT_METRICS

    record = {
        "metric": METRIC_LABELS[metric],
        "source_metric": metric,
        "role": (
            "Forecast target"
            if metric in FORECAST_TARGETS
            else "Auxiliary context"
        ),
        "value_class": (
            "Count/activity"
            if is_count_metric
            else "Performance"
        ),
        "missingness_policy": (
            "Semantic zero"
            if is_count_metric
            else "Preserve missing"
        ),
        "post_freeze_no_cp_rule": (
            "Recursive target state"
            if metric in FORECAST_TARGETS
            else "Ordinary forecasting only"
        ),
    }

    for modeling_track, short_label in coverage_track_order:
        track_rows = multimodal_candidate_state_df.loc[
            multimodal_candidate_state_df["modeling_track"].eq(modeling_track)
        ]

        # Usable predictor coverage AFTER applying the established
        # missing-value semantics.
        record[f"{short_label}_level"] = round(
            float(track_rows[level_col].notna().mean()),
            3,
        )

        record[f"{short_label}_delta"] = round(
            float(track_rows[delta_col].notna().mean()),
            3,
        )

        # Raw source missingness underlying that usable coverage.
        #
        # Count/activity:
        #   Level -> share converted to semantic zero.
        #   Δ1    -> share whose current or prior endpoint used semantic zero.
        #
        # Performance:
        #   Level -> share remaining missing.
        #   Δ1    -> share blocked because one or both endpoints were missing.
        record[f"{short_label}_level_issue"] = round(
            float(
                track_rows[origin_flag_col]
                .fillna(False)
                .astype(bool)
                .mean()
            ),
            3,
        )

        legal_delta_mask = (
            track_rows["_previous_step_available_under_track"]
            .fillna(False)
            .astype(bool)
        )

        legal_delta_flags = (
            track_rows.loc[
                legal_delta_mask,
                delta_flag_col,
            ]
            .dropna()
        )

        record[f"{short_label}_delta_issue"] = (
            round(
                float(
                    legal_delta_flags
                    .astype(bool)
                    .mean()
                ),
                3,
            )
            if len(legal_delta_flags) > 0
            else float("nan")
        )

    coverage_records.append(record)


multimodal_coverage_df = pd.DataFrame(coverage_records)


# ---------------------------------------------------------------------
# Arrange the six ordinary-forecasting coverage views.
# ---------------------------------------------------------------------
coverage_columns = [
    "Post_level",
    "Post_delta",
    "Full_level",
    "Full_delta",
    "Pre_level",
    "Pre_delta",
]

coverage_labels = [
    "Post · Level",
    "Post · Δ1",
    "Full · Level",
    "Full · Δ1",
    "Pre · Level",
    "Pre · Δ1",
]

support_issue_columns = [
    "Post_level_issue",
    "Post_delta_issue",
    "Full_level_issue",
    "Full_delta_issue",
    "Pre_level_issue",
    "Pre_delta_issue",
]


coverage_matrix = (
    multimodal_coverage_df[coverage_columns]
    .to_numpy(dtype=float)
)

support_issue_matrix = (
    multimodal_coverage_df[support_issue_columns]
    .to_numpy(dtype=float)
)


# ---------------------------------------------------------------------
# Plot usable coverage.
#
# Color = share of origins where the predictor is usable after applying
#         the approved missing-value semantics.
#
# Hover = raw missing-input share underneath that coverage.
# ---------------------------------------------------------------------
fig = go.Figure(
    data=go.Heatmap(
        z=coverage_matrix,
        x=coverage_labels,
        y=multimodal_coverage_df["metric"],

        # Show usable coverage numerically inside each cell.
        text=coverage_matrix,
        texttemplate="%{text:.3f}",
        textfont=dict(size=11),

        # Preserve the underlying raw-missingness information for hover.
        customdata=support_issue_matrix,

        zmin=0,
        zmax=1,
        colorscale=[
            [0.00, BRAND_COLORS["ice"]],
            [0.50, BRAND_COLORS["seafoam"]],
            [1.00, BRAND_COLORS["dark_teal"]],
        ],

        hovertemplate=(
            "<b>%{y}</b><br>"
            "%{x}<br>"
            "Usable coverage: %{z:.3f}<br>"
            "Raw missing-input share: %{customdata:.3f}"
            "<extra></extra>"
        ),

        colorbar=dict(
            title="Usable<br>coverage",
            tickformat=".2f",
        ),
    )
)

apply_chart_layout(
    fig,
    title="Usable Multimodal Predictor Coverage After Missingness Handling",
    x_title=None,
    y_title=None,
    height=600,
)

# Give long mobility labels enough room and keep the chart readable in Deepnote.
fig.update_layout(
    margin=dict(
        l=185,
        r=90,
        t=90,
        b=75,
    ),
)

fig.update_yaxes(
    automargin=True,
    tickfont=dict(size=12),
)

fig.update_xaxes(
    tickfont=dict(size=12),
)

fig.show()

# ---------------------------------------------------------------------
# Preserve compact decision metadata for the upcoming HITL decision and
# downstream feature assembly without producing a second reader-facing output.
# ---------------------------------------------------------------------
multimodal_decision_context_df = (
    multimodal_coverage_df[
        [
            "metric",
            "role",
            "value_class",
            "missingness_policy",
            "post_freeze_no_cp_rule",
        ]
    ]
    .copy()
)

**Findings.** After applying the established missing-data rules, all five count/activity measures provide complete usable coverage across the three modeling tracks for both their origin levels and one-step changes. FHVHV speed and duration also remain highly available, with roughly 99.1%–99.5% coverage, while Bus speed remains above 95% throughout. Taxi speed and duration are the main limitation: usable level coverage declines from 84.1% in Post-CP Primary to 73.4% in Full-History and 67.0% in Pre-CP Reference, while one-step-change coverage falls from 78.7% to 64.9% and 56.6%, respectively. The compact multimodal state is therefore broadly usable without additional feature engineering, although Taxi performance context will not be available at every forecast origin and should remain explicitly missing rather than being filled with information learned from future observations.

> 🔶 Decision — Multimodal Forecasting Feature Policy
Decision: retain the compact multimodal design\.

Keep the at\-origin level \+ immediately preceding Daypart change for all 10 mobility measures\. This gives the models a simple view of the broader transportation system without creating a large collection of hand\-built ratios, substitution measures, divergence scores, or other interaction formulas that Tree and neural models can learn from the underlying inputs\.

The multimodal state is built once in a target\-neutral form so it can be reused across all five forecast targets\. This temporarily recreates the target's own level and one\-step change, but those two redundant features will be removed during target\-specific assembly because 4\.2\.1 already provides the target's richer recent\-history representation\.

Retain the project's established missing\-data semantics: missing count/activity observations represent zero activity, while missing speed and duration remain missing\. The resulting coverage is complete for the count/activity features and strong for most performance features, with Taxi speed and duration providing the main partial\-coverage exception\.

For the later no\-CP simulation, context derived from the five forecast targets may be rebuilt from recursively predicted no\-CP mobility after January 4\. The five auxiliary measures remain ordinary\-forecasting\-only unless they are separately modeled\.
Proceed to transportation\-aware spatial context\.

### Section Summary

This section constructed a compact view of the transportation system available at each forecast origin. Ten mobility measures each contribute two candidate features—their current value and their change from the immediately preceding canonical Daypart—producing **20 target-neutral multimodal candidates** that can later be reused across all forecast targets.

The construction follows the project's established interpretation of missing mobility observations. Count/activity measures use zero when no activity was observed, while speed and duration remain missing when performance could not be measured. No global statistical imputation is introduced here. Coverage after applying those rules is complete for all count/activity features, above 95% for Bus speed, and approximately 99% for FHVHV speed and duration. Taxi speed and duration are less consistently available, particularly in the Pre-CP history, but remain usable enough to retain as optional model inputs.

The section also establishes the later forecasting boundary. The five forecast-target measures can be regenerated from recursively predicted mobility during the no-CP simulation, while the five auxiliary measures cannot use actual post-CP observations. During target-specific assembly, the forecast target's own multimodal level and one-step change will be removed because 4.2.1 already provides that series' richer historical feature set.

With the multimodal policy now fixed, the next step is to add **transportation-aware spatial context** describing what was happening in connected Taxi Zones at the same forecast origin.

## 4.2.2.3 Construct Transportation-Aware Spatial Context

Mobility conditions in one Taxi Zone do not occur in isolation. Congestion, travel demand, and transit activity can reflect conditions in nearby neighborhoods as well as locations connected through important transportation corridors.

This section adds a compact spatial view of the transportation system available when each forecast is issued. It uses the established transportation-aware connectivity network, which combines ordinary geographic adjacency with additional transportation connections and geometry fallbacks already validated upstream.

The connectivity network operates at **canonical Taxi Zone geography**, while the forecasting surfaces preserve the physical Taxi Zone identifiers used by the mobility panel. We therefore first reconcile the mobility state to canonical geography before calculating connected-zone conditions. Count/activity measures are summed when more than one source Taxi Zone belongs to the same canonical geography, while speed and duration measures are combined using their associated activity as weights.

As with the multimodal features, all dynamic spatial context is calculated at the **forecast origin**. No mobility from the future target period is used.

**Build canonical mobility state for spatial aggregation.** Convert the dense physical-Taxi-Zone mobility panel into the canonical geography used by the transportation-connectivity network.

The aggregation preserves the missing-data semantics established in the preceding section. Count/activity measures treat missing observations as zero activity before aggregation. Speed and duration remain unavailable when there is no usable activity-weighted observation. This prevents multipart or aliased Taxi Zone identifiers from being counted as independent neighbors while preserving the mobility represented by each source row.

In [12]:
# ---------------------------------------------------------------------
# 4.2.2.3A — Build canonical mobility state for spatial aggregation
# ---------------------------------------------------------------------

# The transportation graph operates on canonical Taxi Zone geography.
SPATIAL_COUNT_METRICS = [
    "taxi_trip_count", "fhvhv_trip_count", "subway_ridership",
    "subway_transfers", "bus_trip_count",
]

SPATIAL_PERFORMANCE_WEIGHT_MAP = {
    "taxi_avg_trip_speed": "taxi_trip_count",
    "taxi_avg_trip_duration": "taxi_trip_count",
    "fhvhv_avg_trip_speed": "fhvhv_trip_count",
    "fhvhv_avg_trip_duration": "fhvhv_trip_count",
    "avg_bus_speed": "bus_trip_count",
}
SPATIAL_PERFORMANCE_METRICS = list(SPATIAL_PERFORMANCE_WEIGHT_MAP)

# The spatial policy must cover exactly the same 10 measures approved above.
spatial_policy_metrics = set(SPATIAL_COUNT_METRICS) | set(SPATIAL_PERFORMANCE_METRICS)
if spatial_policy_metrics != set(MOBILITY_CONTEXT_METRICS):
    raise ValueError(
        "Spatial aggregation policy does not match MOBILITY_CONTEXT_METRICS. "
        f"Policy-only={sorted(spatial_policy_metrics - set(MOBILITY_CONTEXT_METRICS))}; "
        f"context-only={sorted(set(MOBILITY_CONTEXT_METRICS) - spatial_policy_metrics)}"
    )

required_spatial_source_columns = [
    "taxi_zone_id", "canonical_location_id", "observation_sequence_id",
    *MOBILITY_CONTEXT_METRICS,
]
missing_spatial_source_columns = [
    col for col in required_spatial_source_columns if col not in mobility_context_source_df.columns
]

if missing_spatial_source_columns:
    raise ValueError(
        f"Mobility context source is missing required spatial-build columns: "
        f"{missing_spatial_source_columns}"
    )

if mobility_context_source_df["canonical_location_id"].isna().any():
    raise ValueError("Every mobility row must resolve to canonical_location_id.")


# ---------------------------------------------------------------------
# Prepare mobility measures before collapsing physical IDs to canonical IDs
# ---------------------------------------------------------------------

canonical_work_df = mobility_context_source_df[required_spatial_source_columns].copy()

# Count/activity missingness means semantic zero.
for metric in SPATIAL_COUNT_METRICS:
    canonical_work_df[metric] = pd.to_numeric(canonical_work_df[metric], errors="coerce").fillna(0.0)

# Performance values remain missing unless both value and positive activity
# weight are available. Numerator/denominator columns avoid averaging averages.
for metric, weight_metric in SPATIAL_PERFORMANCE_WEIGHT_MAP.items():
    values = pd.to_numeric(canonical_work_df[metric], errors="coerce")
    weights = pd.to_numeric(canonical_work_df[weight_metric], errors="coerce")
    usable = values.notna() & weights.notna() & weights.gt(0)

    canonical_work_df[f"_canonical_num__{metric}"] = (values * weights).where(usable)
    canonical_work_df[f"_canonical_den__{metric}"] = weights.where(usable)


# ---------------------------------------------------------------------
# Collapse to Canonical Taxi Zone × observation_sequence_id
# ---------------------------------------------------------------------

canonical_agg_spec = {metric: (metric, "sum") for metric in SPATIAL_COUNT_METRICS}

for metric in SPATIAL_PERFORMANCE_METRICS:
    canonical_agg_spec[f"_canonical_num__{metric}"] = (f"_canonical_num__{metric}", "sum")
    canonical_agg_spec[f"_canonical_den__{metric}"] = (f"_canonical_den__{metric}", "sum")

canonical_spatial_mobility_state_df = (
    canonical_work_df
    .groupby(["canonical_location_id", "observation_sequence_id"], observed=True, sort=False)
    .agg(**canonical_agg_spec)
    .reset_index()
)

# Recover activity-weighted performance values. Zero denominator => unavailable.
for metric in SPATIAL_PERFORMANCE_METRICS:
    num_col, den_col = f"_canonical_num__{metric}", f"_canonical_den__{metric}"
    denominator = canonical_spatial_mobility_state_df[den_col]
    canonical_spatial_mobility_state_df[metric] = (
        canonical_spatial_mobility_state_df[num_col] / denominator.where(denominator.gt(0))
    )

canonical_spatial_mobility_state_df = (
    canonical_spatial_mobility_state_df[
        ["canonical_location_id", "observation_sequence_id", *MOBILITY_CONTEXT_METRICS]
    ]
    .sort_values(["canonical_location_id", "observation_sequence_id"])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# Validate the canonical spatial state
# ---------------------------------------------------------------------

canonical_state_key = ["canonical_location_id", "observation_sequence_id"]
canonical_state_duplicate_rows = int(
    canonical_spatial_mobility_state_df.duplicated(canonical_state_key, keep=False).sum()
)

if canonical_state_duplicate_rows:
    raise ValueError(
        f"Canonical spatial mobility state contains "
        f"{canonical_state_duplicate_rows:,} duplicate key rows."
    )

source_to_canonical_df = (
    mobility_context_source_df[["taxi_zone_id", "canonical_location_id"]]
    .drop_duplicates()
    .sort_values(["taxi_zone_id", "canonical_location_id"])
    .reset_index(drop=True)
)

if source_to_canonical_df.duplicated("taxi_zone_id").any():
    raise ValueError("A physical Taxi Zone maps to more than one canonical geography.")

canonical_source_counts = source_to_canonical_df.groupby("canonical_location_id")["taxi_zone_id"].nunique()
canonical_zone_count = int(source_to_canonical_df["canonical_location_id"].nunique())
canonical_sequence_count = int(mobility_context_source_df["observation_sequence_id"].nunique())
expected_canonical_state_rows = canonical_zone_count * canonical_sequence_count

if len(canonical_spatial_mobility_state_df) != expected_canonical_state_rows:
    raise ValueError(
        "Canonical spatial state is not dense at Canonical Taxi Zone × observation_sequence_id: "
        f"expected {expected_canonical_state_rows:,}, "
        f"found {len(canonical_spatial_mobility_state_df):,}."
    )

CANONICAL_SPATIAL_MOBILITY_STATE_PATH = (
    INTERMEDIATE_DIR / "canonical_spatial_mobility_state.parquet"
)

if WRITE_INTERMEDIATES:
    canonical_spatial_mobility_state_df.to_parquet(
        CANONICAL_SPATIAL_MOBILITY_STATE_PATH, index=False
    )

canonical_spatial_build_summary_df = pd.DataFrame({
    "Check": [
        "Source mobility rows",
        "Physical Taxi Zone IDs",
        "Canonical spatial geographies",
        "Canonical sequence positions",
        "Expected canonical state rows",
        "Materialized canonical state rows",
        "Canonical geographies with multiple source IDs",
        "Maximum source IDs within one canonical geography",
        "Duplicate canonical-state keys",
    ],
    "Value": [
        len(mobility_context_source_df),
        source_to_canonical_df["taxi_zone_id"].nunique(),
        canonical_zone_count,
        canonical_sequence_count,
        expected_canonical_state_rows,
        len(canonical_spatial_mobility_state_df),
        int((canonical_source_counts > 1).sum()),
        int(canonical_source_counts.max()),
        canonical_state_duplicate_rows,
    ],
})

display(canonical_spatial_build_summary_df)

,Check,Value
0,Source mobility rows,1559590
1,Physical Taxi Zone IDs,263
2,Canonical spatial geographies,262
3,Canonical sequence positions,5930
4,Expected canonical state rows,1553660
5,Materialized canonical state rows,1553660
6,Canonical geographies with multiple source IDs,1
7,Maximum source IDs within one canonical geography,2
8,Duplicate canonical-state keys,0


**Findings.** The physical mobility panel collapses cleanly to the canonical geography used by the transportation-connectivity network. The 263 physical Taxi Zone IDs resolve to 262 canonical spatial geographies across all 5,930 sequence positions, producing exactly the expected 1,553,660 canonical-state rows with no duplicate keys. Only one canonical geography combines multiple physical source IDs, and that case contains just two IDs. The canonicalization step therefore changes the geography only where required and preserves a complete, unique mobility state for downstream connected-zone aggregation.

**Construct transportation-connected origin context.** For each canonical Taxi Zone and sequence position, summarize the mobility state of all Taxi Zones linked to it by the established transportation network.

Count/activity measures use the **mean across connected geographies**, so zones with more network connections do not automatically receive larger feature values. Speed and duration use **activity-weighted connected averages**, so a very low-activity neighbor does not contribute the same influence as a high-activity neighbor. Connectivity type is retained as provenance upstream but is not split into separate feature families here; all validated transportation links contribute to one connected context.

Observed-neighbor counts and shares are retained as support metadata for performance measures. They are useful for understanding feature availability but are not yet treated as model predictors.

In [13]:
# ---------------------------------------------------------------------
# 4.2.2.3B — Construct transportation-connected origin context
# ---------------------------------------------------------------------

def connected_feature_column(metric: str) -> str:
    """Return the production feature name for one connected mobility measure."""
    return f"spatial_connected__{metric}"


def connected_support_count_column(metric: str) -> str:
    """Return the QA field counting connected zones with usable performance."""
    return f"spatial_connected_support_count__{metric}"


def connected_support_share_column(metric: str) -> str:
    """Return the QA field measuring usable connected-zone performance coverage."""
    return f"spatial_connected_support_share__{metric}"


# One production connected-context feature is retained for each of the same
# 10 mobility measures used in the local multimodal state.
SPATIAL_CONNECTED_FEATURE_COLUMNS = [
    connected_feature_column(metric) for metric in MOBILITY_CONTEXT_METRICS
]


# ---------------------------------------------------------------------
# Normalize the authoritative transportation network
# ---------------------------------------------------------------------

# The upstream connectivity table may retain descriptive/provenance columns,
# but dynamic aggregation needs only the canonical source → connected pair.
# Canonical geography is essential here because the graph was constructed
# after harmonizing physical Taxi Zone IDs.
spatial_edge_pairs_df = (
    taxi_zone_connectivity_df[
        ["canonical_location_id", "connected_canonical_location_id"]
    ]
    .drop_duplicates()
    .sort_values(["canonical_location_id", "connected_canonical_location_id"])
    .reset_index(drop=True)
)

if spatial_edge_pairs_df.empty:
    raise ValueError("Transportation-connectivity table contains no usable canonical pairs.")

# A source zone should never be counted as its own connected neighbor.
self_loop_count = int(
    spatial_edge_pairs_df["canonical_location_id"]
    .eq(spatial_edge_pairs_df["connected_canonical_location_id"])
    .sum()
)

if self_loop_count:
    raise ValueError(
        f"Transportation-connectivity table contains {self_loop_count:,} self-loop pairs."
    )


# Every connected geography must resolve to the canonical mobility state built
# in 3A. Otherwise a graph edge would point to a geography whose mobility state
# cannot be observed at the forecast origin.
canonical_state_ids = set(
    canonical_spatial_mobility_state_df["canonical_location_id"]
    .dropna()
    .astype(int)
)

connected_ids = set(
    spatial_edge_pairs_df["connected_canonical_location_id"]
    .dropna()
    .astype(int)
)

missing_connected_ids = sorted(connected_ids - canonical_state_ids)

if missing_connected_ids:
    raise ValueError(
        "Connected canonical geographies are missing from the mobility state: "
        f"{missing_connected_ids}"
    )


# Preserve each source geography's number of network neighbors as QA/support
# metadata. This is not one of the 10 selected dynamic model predictors.
network_neighbor_count_df = (
    spatial_edge_pairs_df
    .groupby("canonical_location_id", as_index=False)
    .agg(
        spatial_network_neighbor_count=(
            "connected_canonical_location_id", "nunique"
        )
    )
)


# ---------------------------------------------------------------------
# Expand each network edge across the canonical mobility timeline
# ---------------------------------------------------------------------

# Rename mobility columns before the join so it remains visually clear that
# these values belong to the CONNECTED geography rather than the focal source
# geography whose context we are constructing.
neighbor_column_map = {
    metric: f"_neighbor__{metric}"
    for metric in MOBILITY_CONTEXT_METRICS
}

neighbor_mobility_state_df = (
    canonical_spatial_mobility_state_df[
        [
            "canonical_location_id",
            "observation_sequence_id",
            *MOBILITY_CONTEXT_METRICS,
        ]
    ]
    .rename(columns={
        "canonical_location_id": "connected_canonical_location_id",
        **neighbor_column_map,
    })
)

# Each source→connected edge receives the connected geography's state at every
# canonical sequence position. With 876 network pairs and 5,930 positions,
# this deliberately creates the edge × time working surface used below.
edge_neighbor_state_df = spatial_edge_pairs_df.merge(
    neighbor_mobility_state_df,
    on="connected_canonical_location_id",
    how="left",
    validate="many_to_many",
)

expected_edge_state_rows = len(spatial_edge_pairs_df) * canonical_sequence_count

if len(edge_neighbor_state_df) != expected_edge_state_rows:
    raise ValueError(
        "Connected-state expansion has an unexpected row count: "
        f"expected {expected_edge_state_rows:,}, "
        f"found {len(edge_neighbor_state_df):,}."
    )


# ---------------------------------------------------------------------
# Aggregate connected-zone mobility to Source Canonical Zone × Origin
# ---------------------------------------------------------------------

# Count/activity measures already obey the established semantic-zero policy
# from 3A. Their connected context is therefore the mean state across all
# transportation-connected canonical geographies, including legitimate zeros.
connected_agg_spec = {
    connected_feature_column(metric): (neighbor_column_map[metric], "mean")
    for metric in SPATIAL_COUNT_METRICS
}


# Performance measures require different treatment. A connected zone with no
# observed speed/duration must NOT become artificial zero performance. Instead,
# combine only usable performance observations and weight them by the matching
# activity measure (Taxi trips, FHVHV trips, or Bus trips).
for metric, weight_metric in SPATIAL_PERFORMANCE_WEIGHT_MAP.items():
    value_col = neighbor_column_map[metric]
    weight_col = neighbor_column_map[weight_metric]

    num_col = f"_connected_num__{metric}"
    den_col = f"_connected_den__{metric}"
    support_col = f"_connected_support__{metric}"

    values = pd.to_numeric(edge_neighbor_state_df[value_col], errors="coerce")
    weights = pd.to_numeric(edge_neighbor_state_df[weight_col], errors="coerce")

    # Performance contributes only when both the performance value and a
    # positive activity weight exist. This prevents inactive zones from
    # manufacturing zero speed or duration.
    usable = values.notna() & weights.notna() & weights.gt(0)

    edge_neighbor_state_df[num_col] = (values * weights).where(usable)
    edge_neighbor_state_df[den_col] = weights.where(usable)
    edge_neighbor_state_df[support_col] = usable.astype("int8")

    # Aggregate numerators/denominators separately so we never average averages.
    connected_agg_spec[num_col] = (num_col, "sum")
    connected_agg_spec[den_col] = (den_col, "sum")

    # Support counts remain QA metadata describing how many connected zones
    # supplied an actual performance observation at this origin.
    connected_agg_spec[connected_support_count_column(metric)] = (
        support_col, "sum"
    )


connected_canonical_state_df = (
    edge_neighbor_state_df
    .groupby(
        ["canonical_location_id", "observation_sequence_id"],
        observed=True,
        sort=False,
    )
    .agg(**connected_agg_spec)
    .reset_index()
    .merge(
        network_neighbor_count_df,
        on="canonical_location_id",
        how="left",
        validate="many_to_one",
    )
)


# Recover the activity-weighted performance features after aggregation.
# A zero denominator means that none of the connected geographies supplied
# usable performance and therefore remains missing rather than becoming zero.
for metric in SPATIAL_PERFORMANCE_METRICS:
    num_col = f"_connected_num__{metric}"
    den_col = f"_connected_den__{metric}"
    denominator = connected_canonical_state_df[den_col]

    connected_canonical_state_df[connected_feature_column(metric)] = (
        connected_canonical_state_df[num_col]
        / denominator.where(denominator.gt(0))
    )

    # Support share makes the performance coverage interpretable later without
    # promoting support information into the actual predictor family.
    connected_canonical_state_df[connected_support_share_column(metric)] = (
        connected_canonical_state_df[connected_support_count_column(metric)]
        / connected_canonical_state_df["spatial_network_neighbor_count"].where(
            connected_canonical_state_df["spatial_network_neighbor_count"].gt(0)
        )
    )


# Temporary weighted numerators/denominators have served their purpose and
# should not leak into later feature-selection logic.
connected_temp_columns = [
    column
    for column in connected_canonical_state_df.columns
    if (
        column.startswith("_connected_num__")
        or column.startswith("_connected_den__")
    )
]

connected_canonical_state_df = (
    connected_canonical_state_df
    .drop(columns=connected_temp_columns)
    .sort_values(["canonical_location_id", "observation_sequence_id"])
    .reset_index(drop=True)
)

if connected_canonical_state_df.duplicated(
    ["canonical_location_id", "observation_sequence_id"]
).any():
    raise ValueError(
        "Connected canonical state is not unique at "
        "canonical_location_id × observation_sequence_id."
    )


# ---------------------------------------------------------------------
# Map canonical connected context back to physical forecast origins
# ---------------------------------------------------------------------

# Forecasting operates on physical Taxi Zone sequences, whereas the graph
# operates on canonical geography. Bridge the two only AFTER connected-state
# aggregation so canonicalized zones do not double-count neighbors.
origin_canonical_bridge_df = (
    forecast_physical_origin_df[PHYSICAL_ORIGIN_KEYS]
    .merge(
        mobility_zone_lookup_df[
            ["taxi_zone_id", "canonical_location_id"]
        ],
        on="taxi_zone_id",
        how="left",
        validate="many_to_one",
    )
)

if origin_canonical_bridge_df["canonical_location_id"].isna().any():
    unresolved_origins = int(
        origin_canonical_bridge_df["canonical_location_id"].isna().sum()
    )
    raise ValueError(
        f"{unresolved_origins:,} forecasting origins do not resolve "
        "to canonical geography."
    )


# THIS is the object consumed by 3C, 3D, and later 3E.
spatial_connected_origin_state_df = origin_canonical_bridge_df.merge(
    connected_canonical_state_df,
    on=["canonical_location_id", "observation_sequence_id"],
    how="left",
    validate="many_to_one",
)


# A Taxi Zone with no graph edge has zero network neighbors, but its dynamic
# connected features stay missing. Zero neighbors does NOT mean zero connected
# mobility.
spatial_connected_origin_state_df["spatial_network_neighbor_count"] = (
    spatial_connected_origin_state_df["spatial_network_neighbor_count"]
    .fillna(0)
    .astype("int16")
)

# Performance support count is legitimately zero when no connected geography
# supplied usable performance. Support shares remain NaN when there is no
# connected network denominator at all.
for metric in SPATIAL_PERFORMANCE_METRICS:
    support_col = connected_support_count_column(metric)

    spatial_connected_origin_state_df[support_col] = (
        spatial_connected_origin_state_df[support_col]
        .fillna(0)
        .astype("int16")
    )


# The mapping back to physical geography must restore exactly one connected
# context row per native physical forecast origin.
if spatial_connected_origin_state_df.duplicated(PHYSICAL_ORIGIN_KEYS).any():
    duplicate_rows = int(
        spatial_connected_origin_state_df
        .duplicated(PHYSICAL_ORIGIN_KEYS, keep=False)
        .sum()
    )
    raise ValueError(
        "Spatial connected-origin state is not unique at the physical-origin "
        f"grain: {duplicate_rows:,} duplicate rows."
    )


# Persist the expensive connected-state construction when intermediate writes
# are enabled so later development does not require rebuilding the full
# edge × sequence working surface.
CONNECTED_SPATIAL_ORIGIN_STATE_PATH = (
    INTERMEDIATE_DIR / "connected_spatial_origin_state.parquet"
)

if WRITE_INTERMEDIATES:
    spatial_connected_origin_state_df.to_parquet(
        CONNECTED_SPATIAL_ORIGIN_STATE_PATH,
        index=False,
    )


# ---------------------------------------------------------------------
# Compact reader-facing validation
# ---------------------------------------------------------------------

# Summarize the actual native track populations after spatial attachment.
# The key question is whether connected context exists—not whether every
# performance metric is non-null, which 3C examines separately.
spatial_connected_build_summary_df = (
    spatial_connected_origin_state_df
    .groupby("modeling_track", observed=True)
    .agg(
        origin_rows=("taxi_zone_id", "size"),
        taxi_zones=("taxi_zone_id", "nunique"),
        origins_with_connected_context_share=(
            "spatial_network_neighbor_count",
            lambda values: float(values.gt(0).mean()),
        ),
        mean_network_neighbors=("spatial_network_neighbor_count", "mean"),
        median_network_neighbors=("spatial_network_neighbor_count", "median"),
    )
    .reset_index()
)

spatial_connected_build_summary_df["connected_feature_count"] = len(
    SPATIAL_CONNECTED_FEATURE_COLUMNS
)

# We already fail above if duplicate keys exist, but exposing zero here gives
# the notebook reader an immediate grain-level QA checkpoint.
spatial_connected_build_summary_df["duplicate_origin_keys"] = 0

round_columns = [
    "origins_with_connected_context_share",
    "mean_network_neighbors",
    "median_network_neighbors",
]

spatial_connected_build_summary_df[round_columns] = (
    spatial_connected_build_summary_df[round_columns].round(3)
)

display(spatial_connected_build_summary_df)

,modeling_track,origin_rows,taxi_zones,origins_with_connected_context_share,mean_network_neighbors,median_network_neighbors,connected_feature_count,duplicate_origin_keys
0,full_history_regime_aware,1529682,258,0.992,3.349,3.000,10,0
1,post_cp_primary,579278,257,0.992,3.358,3.000,10,0
2,pre_cp_reference,948150,258,0.992,3.349,3.000,10,0


**Findings.** The transportation-connected feature surface preserves the full native forecasting populations for all three modeling tracks with zero duplicate origin keys. Approximately 99.2% of forecast origins resolve to at least one transportation-connected geography, with an average of about 3.35 connected zones and a median of 3 across every track. The resulting spatial layer contributes exactly 10 connected-context candidates while leaving the small share of origins without network connections explicitly unsupported rather than assigning artificial spillover values.

**Visualize connected-feature availability.** The spatial features are useful only where the transportation network and the underlying mobility measure provide enough information at the forecast origin.

This chart measures the share of physical forecast origins with a usable connected-zone value for each mobility measure. Count/activity coverage should mainly reflect whether a canonical geography has transportation connections; performance coverage can be lower because connected speed or duration also requires usable activity-weighted observations among those neighbors.

This remains an availability diagnostic, not a test of predictive value.

In [14]:
# ---------------------------------------------------------------------
# 4.2.2.3C — Visualize connected-feature availability
# ---------------------------------------------------------------------

SPATIAL_TRACK_ORDER = [
    "post_cp_primary",
    "full_history_regime_aware",
    "pre_cp_reference",
]

SPATIAL_TRACK_LABELS = {
    "post_cp_primary": "Post-CP Primary",
    "full_history_regime_aware": "Full-History Regime-Aware",
    "pre_cp_reference": "Pre-CP Reference",
}

spatial_coverage_records = []

for modeling_track in SPATIAL_TRACK_ORDER:
    track_df = spatial_connected_origin_state_df.loc[
        spatial_connected_origin_state_df["modeling_track"].eq(modeling_track)
    ]

    for metric in MOBILITY_CONTEXT_METRICS:
        spatial_coverage_records.append({
            "modeling_track": modeling_track,
            "track_label": SPATIAL_TRACK_LABELS[modeling_track],
            "metric": metric,
            "metric_label": METRIC_LABELS.get(metric, metric),
            "usable_coverage": float(
                track_df[connected_feature_column(metric)].notna().mean()
            ),
        })

spatial_connected_coverage_df = pd.DataFrame(spatial_coverage_records)

metric_display_order = [
    METRIC_LABELS.get(metric, metric) for metric in MOBILITY_CONTEXT_METRICS
]
track_display_order = [
    SPATIAL_TRACK_LABELS[track] for track in SPATIAL_TRACK_ORDER
]

spatial_coverage_heatmap_df = (
    spatial_connected_coverage_df
    .pivot(index="metric_label", columns="track_label", values="usable_coverage")
    .reindex(index=metric_display_order, columns=track_display_order)
)

spatial_coverage_fig = px.imshow(
    spatial_coverage_heatmap_df,
    text_auto=".3f",
    aspect="auto",
    zmin=0,
    zmax=1,
    color_continuous_scale=[BRAND_COLORS["ice"], BRAND_COLORS["dark_teal"]],
)

spatial_coverage_fig.update_traces(
    hovertemplate=(
        "Metric=%{y}<br>"
        "Modeling track=%{x}<br>"
        "Usable coverage=%{z:.3f}"
        "<extra></extra>"
    )
)

spatial_coverage_fig.update_layout(
    title="Usable Transportation-Connected Context Coverage",
    width=920,
    height=560,
    xaxis_title="Modeling track",
    yaxis_title="Mobility measure",
    margin=dict(l=170, r=40, t=75, b=60),
)

spatial_coverage_fig.update_coloraxes(colorbar_title="Coverage")
spatial_coverage_fig.show()

**Findings.** Transportation-connected conditions are related to local mobility but usually provide meaningfully different information. Local-versus-connected Spearman correlations are moderate for Subway ridership (about 0.50), FHVHV trips (about 0.54), Bus trips (about 0.57), and especially Subway transfers (about 0.36), indicating substantial variation between a Taxi Zone and its connected transportation environment. Taxi trips remain more strongly related at roughly 0.74–0.77, while Taxi speed and duration become noticeably less similar in the longer historical tracks. FHVHV speed is the strongest exception, with correlations near 0.92 in every track, suggesting considerable redundancy for that individual measure. Overall, however, the connected state is far from a simple copy of local conditions and provides a distinct spatial view of the transportation system.

**Check whether connected conditions add distinct spatial information.** High coverage alone does not tell us whether connected-zone state adds anything beyond the local Taxi Zone state already available to the model.

For each mobility measure, compare the local origin value with the transportation-connected value at the same origin. A correlation near 1 means the connected feature tends to move almost exactly with the local measure; a lower correlation indicates that connected areas often provide a different view of system conditions.

This is deliberately a structural diagnostic rather than a model-performance test. A deterministic sample keeps the calculation practical while preserving the full production feature surfaces.

In [15]:
# ---------------------------------------------------------------------
# 4.2.2.3D — Check whether connected conditions add distinct spatial information
# ---------------------------------------------------------------------

SPATIAL_DIAGNOSTIC_SAMPLE_SIZE = 150_000
SPATIAL_DIAGNOSTIC_RANDOM_STATE = 696

required_local_context_columns = [*PHYSICAL_ORIGIN_KEYS, *MOBILITY_CONTEXT_METRICS]
missing_local_context_columns = [
    col for col in required_local_context_columns if col not in origin_mobility_context_df.columns
]

if missing_local_context_columns:
    raise ValueError(
        "Origin mobility context is missing columns required for the spatial "
        f"similarity diagnostic: {missing_local_context_columns}"
    )

local_context_diagnostic_df = (
    origin_mobility_context_df[required_local_context_columns]
    .merge(
        spatial_connected_origin_state_df[
            [*PHYSICAL_ORIGIN_KEYS, *SPATIAL_CONNECTED_FEATURE_COLUMNS]
        ],
        on=PHYSICAL_ORIGIN_KEYS,
        how="inner",
        validate="one_to_one",
    )
)

if len(local_context_diagnostic_df) != len(spatial_connected_origin_state_df):
    raise ValueError(
        "Local and connected spatial diagnostic surfaces did not align "
        "one-to-one at the physical-origin grain."
    )


# ---------------------------------------------------------------------
# Compare local and connected state within each modeling track
# ---------------------------------------------------------------------

spatial_similarity_records = []

for modeling_track in SPATIAL_TRACK_ORDER:
    track_df = local_context_diagnostic_df.loc[
        local_context_diagnostic_df["modeling_track"].eq(modeling_track)
    ].copy()

    # Reuse one deterministic origin sample across all metrics in each track.
    if len(track_df) > SPATIAL_DIAGNOSTIC_SAMPLE_SIZE:
        track_df = track_df.sample(
            n=SPATIAL_DIAGNOSTIC_SAMPLE_SIZE,
            random_state=SPATIAL_DIAGNOSTIC_RANDOM_STATE,
        )

    for metric in MOBILITY_CONTEXT_METRICS:
        local_values = pd.to_numeric(track_df[metric], errors="coerce")
        connected_values = pd.to_numeric(
            track_df[connected_feature_column(metric)], errors="coerce"
        )

        # Count/activity missingness retains the approved semantic-zero meaning.
        if metric in SPATIAL_COUNT_METRICS:
            local_values = local_values.fillna(0.0)

        usable = local_values.notna() & connected_values.notna()
        paired_rows = int(usable.sum())

        # Spearman correlation = Pearson correlation between ranked vectors.
        if paired_rows >= 30:
            local_rank = local_values.loc[usable].rank(method="average")
            connected_rank = connected_values.loc[usable].rank(method="average")
            spearman_correlation = float(local_rank.corr(connected_rank))
        else:
            spearman_correlation = np.nan

        spatial_similarity_records.append({
            "modeling_track": modeling_track,
            "track_label": SPATIAL_TRACK_LABELS[modeling_track],
            "metric": metric,
            "metric_label": METRIC_LABELS.get(metric, metric),
            "sample_rows": len(track_df),
            "paired_rows": paired_rows,
            "paired_share": paired_rows / len(track_df) if len(track_df) else np.nan,
            "local_connected_spearman": spearman_correlation,
        })

spatial_local_connected_similarity_df = pd.DataFrame(spatial_similarity_records)

spatial_similarity_heatmap_df = (
    spatial_local_connected_similarity_df
    .pivot(index="metric_label", columns="track_label", values="local_connected_spearman")
    .reindex(index=metric_display_order, columns=track_display_order)
)

spatial_similarity_fig = px.imshow(
    spatial_similarity_heatmap_df,
    text_auto=".3f",
    aspect="auto",
    zmin=-1,
    zmax=1,
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
)

spatial_similarity_fig.update_traces(
    hovertemplate=(
        "Metric=%{y}<br>"
        "Modeling track=%{x}<br>"
        "Local-connected Spearman=%{z:.3f}"
        "<extra></extra>"
    )
)

spatial_similarity_fig.update_layout(
    title="Similarity of Local and Transportation-Connected Mobility State",
    width=920,
    height=560,
    xaxis_title="Modeling track",
    yaxis_title="Mobility measure",
    margin=dict(l=170, r=40, t=75, b=60),
)

spatial_similarity_fig.update_coloraxes(colorbar_title="Spearman")
spatial_similarity_fig.show()

**Findings.** Transportation-connected conditions are related to local mobility but usually provide meaningfully different information. Local-versus-connected Spearman correlations are moderate for Subway ridership (about 0.50), FHVHV trips (about 0.54), Bus trips (about 0.57), and especially Subway transfers (about 0.36), indicating substantial variation between a Taxi Zone and its connected transportation environment. Taxi trips remain more strongly related at roughly 0.74–0.77, while Taxi speed and duration become noticeably less similar in the longer historical tracks. FHVHV speed is the strongest exception, with correlations near 0.92 in every track, suggesting considerable redundancy for that individual measure. Overall, however, the connected state is far from a simple copy of local conditions and provides a distinct spatial view of the transportation system.

> 🔶 Decision — Spatial Forecasting Feature Scope
Recommended decision: retain the compact 10\-feature transportation\-connected context layer\.

Coverage is exceptionally strong: approximately 99\.2% of forecast origins have transportation\-connected context, and most individual measures retain usable coverage above 98%\. Taxi speed and duration are less complete—particularly in the Pre\-CP history—but still retain enough coverage to remain useful candidates without introducing leakage\-prone or global imputation\.

The connected features also provide genuinely different information from the local Taxi Zone state\. Most local\-versus\-connected correlations fall in the moderate range, with particularly distinct spatial behavior for Subway transfers, Subway ridership, FHVHV trips, and Bus trips\. FHVHV speed is much more redundant at roughly 0\.92, but one highly correlated measure is not enough to justify complicating an otherwise small 10\-feature family\.

Keep the three validated connectivity types combined into a single transportation network\. Keep performance\-support counts and shares as QA metadata rather than model predictors\. Do not add borough\-wide context or handcrafted local\-minus\-connected differences at this stage; downstream models will already receive both the local and connected states and can learn those relationships directly\.

Decision: proceed with the 10 connected\-context features and move on to static geography and forecast/counterfactual availability rules\.

**Attach inherited static spatial context.** The dynamic connected-zone features describe what was happening around each Taxi Zone at a forecast origin. We now complete the spatial representation with the stable geographic and policy characteristics already maintained in the authoritative Taxi Zone reference.

No new static geography is engineered here. We simply attach the established borough, CBD relationship, distance, and network-position fields to the same physical origin state used by the connected-zone features.

These attributes do not change over time, so they are available to every forecasting track without creating a temporal leakage problem. Together, the two spatial layers answer complementary questions: dynamic connected context describes **what was happening around the zone**, while static context describes **where the zone sits in the city's transportation and congestion-pricing geography**.

In [16]:
# 4.2.2.3E — Attach inherited static spatial context

# Retain only static attributes that describe geography, policy exposure, or
# transportation-network position. Geometry, centroids, duplicate distance
# units, and descriptive identifiers remain metadata rather than predictors.
STATIC_CONTEXT_PREDICTOR_COLUMNS = [
    "borough",
    "is_cbd_zone", "is_cbd_gateway_zone", "is_cbd_adjacent_zone",
    "cbd_spatial_category",
    "distance_to_cbd_miles", "distance_to_gateway_miles",
    "connected_neighbor_count",
    "network_betweenness_centrality",
    "network_closeness_centrality",
    "network_clustering_coefficient",
]

STATIC_CONTEXT_METADATA_COLUMNS = ["canonical_location_id", "zone"]

STATIC_CONTEXT_LABELS = {
    "borough": "Borough",
    "is_cbd_zone": "CBD zone",
    "is_cbd_gateway_zone": "CBD gateway zone",
    "is_cbd_adjacent_zone": "Adjacent to CBD",
    "cbd_spatial_category": "CBD spatial category",
    "distance_to_cbd_miles": "Distance to CBD",
    "distance_to_gateway_miles": "Distance to CBD gateway",
    "connected_neighbor_count": "Connected-neighbor count",
    "network_betweenness_centrality": "Network betweenness",
    "network_closeness_centrality": "Network closeness",
    "network_clustering_coefficient": "Network clustering",
}

# Confirm that the authoritative Taxi Zone reference actually contains every
# field required by the selected static-context contract before joining it to
# the much larger forecasting-origin surface.
required_static_columns = ["taxi_zone_id"] + STATIC_CONTEXT_METADATA_COLUMNS + STATIC_CONTEXT_PREDICTOR_COLUMNS
missing_static_columns = sorted(set(required_static_columns).difference(taxi_zone_context_df.columns))

if missing_static_columns:
    raise ValueError(f"Static Taxi Zone context is missing required columns: {missing_static_columns}")

# Restrict the reference to Taxi Zones that appear in at least one native
# forecasting population. The authoritative source may legitimately contain
# additional zones that never appear in a given forecasting track.
forecast_zone_ids = pd.Index(forecast_physical_origin_df["taxi_zone_id"].dropna().unique())

forecast_static_context_df = (
    taxi_zone_context_df.loc[
        taxi_zone_context_df["taxi_zone_id"].isin(forecast_zone_ids),
        required_static_columns,
    ]
    .copy()
)

validate_unique_key(
    forecast_static_context_df,
    ["taxi_zone_id"],
    "Forecast static Taxi Zone context",
)

# A missing crosswalk here would mean some forecasting origins could never
# receive static spatial context, so fail before expanding across millions of
# origin rows.
missing_forecast_zones = forecast_zone_ids.difference(forecast_static_context_df["taxi_zone_id"])

if len(missing_forecast_zones):
    raise ValueError(
        "Static spatial context is missing forecasting Taxi Zones: "
        f"{missing_forecast_zones.tolist()}"
    )

# Some identity fields may already exist on the connected-origin surface.
# Attach only columns that are not already present so we do not create
# accidental _x/_y duplicates of authoritative geography.
static_attach_columns = [
    column
    for column in STATIC_CONTEXT_METADATA_COLUMNS + STATIC_CONTEXT_PREDICTOR_COLUMNS
    if column not in spatial_connected_origin_state_df.columns
]

spatial_context_origin_state_df = spatial_connected_origin_state_df.merge(
    forecast_static_context_df[["taxi_zone_id"] + static_attach_columns],
    on="taxi_zone_id",
    how="left",
    validate="many_to_one",
)

# The combined spatial object must still contain exactly one row per native
# Modeling Track × Taxi Zone × origin. Static attachment must never change that
# grain or multiply rows.
validate_unique_key(
    spatial_context_origin_state_df,
    PHYSICAL_ORIGIN_KEYS,
    "Combined dynamic and static spatial origin context",
)

# ---------------------------------------------------------------------
# Validate static coverage across genuine mapped Taxi Zones
# ---------------------------------------------------------------------

# TLC location IDs 264 and 265 are deliberate "Unknown" placeholders carried
# upstream to preserve lossless joins. They are valid forecasting identifiers,
# but they do not represent mapped Taxi Zone geography and therefore should not
# be counted as failures of the static spatial-context layer.
UNKNOWN_TAXI_ZONE_IDS = {264, 265}

# Static values repeat across every origin, so evaluate coverage once per Taxi
# Zone rather than letting repeated temporal rows dominate the diagnostic.
static_zone_qa_df = (
    spatial_context_origin_state_df[
        ["taxi_zone_id", "zone"] + STATIC_CONTEXT_PREDICTOR_COLUMNS
    ]
    .drop_duplicates("taxi_zone_id")
)

unknown_static_zone_qa_df = static_zone_qa_df.loc[
    static_zone_qa_df["taxi_zone_id"].isin(UNKNOWN_TAXI_ZONE_IDS)
].copy()

mapped_static_zone_qa_df = static_zone_qa_df.loc[
    ~static_zone_qa_df["taxi_zone_id"].isin(UNKNOWN_TAXI_ZONE_IDS)
].copy()

# Missing spatial attributes are acceptable for the two explicit Unknown
# placeholders only. Any missing value among mapped Taxi Zones would indicate
# a genuine reference-layer or join problem and should stop the notebook.
unexpected_mapped_missing = (
    mapped_static_zone_qa_df[STATIC_CONTEXT_PREDICTOR_COLUMNS]
    .isna()
    .any(axis=1)
)

if unexpected_mapped_missing.any():
    problem_zones = (
        mapped_static_zone_qa_df.loc[
            unexpected_mapped_missing,
            ["taxi_zone_id", "zone"],
        ]
        .to_dict("records")
    )
    raise ValueError(
        "Mapped forecasting Taxi Zones contain unexpected missing static "
        f"context: {problem_zones}"
    )

# Report coverage against the population for which spatial attributes are
# actually defined, while keeping the preserved Unknown placeholders visible
# so the denominator adjustment is explicit rather than hidden.
static_context_coverage_df = pd.DataFrame([
    {
        "Static feature": STATIC_CONTEXT_LABELS[column],
        "Mapped forecast zones": len(mapped_static_zone_qa_df),
        "Zones with value": int(mapped_static_zone_qa_df[column].notna().sum()),
        "Coverage share": round(
            float(mapped_static_zone_qa_df[column].notna().mean()), 3
        ),
        "Preserved Unknown placeholders": len(unknown_static_zone_qa_df),
    }
    for column in STATIC_CONTEXT_PREDICTOR_COLUMNS
])

display(static_context_coverage_df)

,Static feature,Mapped forecast zones,Zones with value,Coverage share,Preserved Unknown placeholders
0,Borough,256,256,1.000,2
1,CBD zone,256,256,1.000,2
2,CBD gateway zone,256,256,1.000,2
3,Adjacent to CBD,256,256,1.000,2
4,CBD spatial category,256,256,1.000,2
5,Distance to CBD,256,256,1.000,2
6,Distance to CBD gateway,256,256,1.000,2
7,Connected-neighbor count,256,256,1.000,2
8,Network betweenness,256,256,1.000,2
9,Network closeness,256,256,1.000,2


**Findings.** Static geographic and policy context is complete for every spatially mapped Taxi Zone in the forecasting population. The only records without mapped spatial attributes are Taxi Zone IDs 264 and 265, the two deliberate "Unknown" TLC placeholders retained upstream for lossless joins; they remain in the forecasting surfaces with spatial context left missing rather than being assigned artificial geography. Excluding those non-spatial placeholders from the coverage denominator, all 11 retained static predictors have complete coverage across mapped forecast zones. The static layer can therefore be carried forward without additional imputation or spatial feature engineering.

### Section Summary

This section builds a compact spatial description of each forecast origin from two complementary views. The transportation-connected layer summarizes mobility conditions across each Taxi Zone's validated transportation network, while the inherited static layer describes the zone's stable geographic, policy, and network position.

The connected context is highly available and, for most measures, meaningfully different from the local mobility state. We therefore retain one connected-zone feature for each of the 10 mobility measures without adding borough-wide dynamic averages, connectivity-type-specific variants, or handcrafted local-minus-neighbor differences. The static attributes are reused directly from the authoritative upstream Taxi Zone reference rather than re-engineered here.

Together, these features give downstream models both a dynamic view of **what was happening around a Taxi Zone** and a stable description of **where that zone sits in the transportation system**.

## 4.2.2.4 Assemble and Validate Multimodal and Spatial Context Features

The multimodal and spatial feature families are now defined. The final step is to turn them into a forecasting handoff that follows the exact origin structure established in 4.2.1.

Before joining anything, we first make the availability rules explicit. Ordinary forecasting may use transportation conditions observed through the forecast origin. The later no-congestion-pricing simulation is stricter: after the January 4, 2025 freeze, actual post-CP mobility cannot update a hypothetical world in which congestion pricing never began.

Dynamic context derived from the five forecast targets can still be rebuilt from recursively predicted no-CP mobility. This includes connected-zone summaries once the recursively predicted states of the required zones are available. The five auxiliary mobility measures are not forecast targets, so their dynamic context becomes unavailable after the freeze unless they are separately modeled. Static geography remains available throughout.

The resulting 4.2.2 exports remain **origin-context surfaces**, not the final h=1, h=2, and h=5 modeling matrices. Weather and the final horizon-specific feature assembly are deferred to 4.2.3.

**Define the context availability and runtime contract.** Record how every selected 4.2.2 predictor behaves under ordinary forecasting and under the frozen Pre-CP no-CP simulation.

The same catalog also records one target-specific exclusion established earlier in the notebook. The target-neutral multimodal state temporarily contains every mobility measure, but once a Forecast Sequence is known, the forecast target's own local level and one-step change are excluded because 4.2.1 already provides a richer representation of that series' history. Its transportation-connected value remains eligible because it describes surrounding mobility rather than the target's own local history.

In [17]:
# 4.2.2.4A — Define the context availability and runtime contract

# Auxiliary measures can improve ordinary forecasts but are not themselves
# forecast targets. Derive this list from the already-approved 10-measure
# context contract rather than maintaining a second hard-coded copy.
AUXILIARY_CONTEXT_METRICS = [
    metric for metric in MOBILITY_CONTEXT_METRICS
    if metric not in FORECAST_TARGETS
]


def context_value_class(metric):
    """Return the established semantic class used for missing-value handling."""
    return "count/activity" if metric in SPATIAL_COUNT_METRICS else "performance"


def context_missing_policy(metric):
    """Describe how missing values were treated when the feature was constructed."""
    if metric in SPATIAL_COUNT_METRICS:
        return "Missing observation represents semantic zero activity."
    return (
        "Preserve missing performance; any later numeric imputation "
        "must be fitted within the legal model-training window."
    )


# ---------------------------------------------------------------------
# 1. Local multimodal origin state
# ---------------------------------------------------------------------

context_catalog_records = []

for metric in MOBILITY_CONTEXT_METRICS:
    is_target_source = metric in FORECAST_TARGETS

    # Target-derived dynamic state can be regenerated after the Pre-CP freeze
    # because those five mobility measures are explicitly forecast. Auxiliary
    # measures cannot legally read their actual post-CP observations.
    post_freeze_class = "recursive_required" if is_target_source else "ordinary_only"
    runtime_source = (
        "recursive_local_target_state"
        if is_target_source
        else "unavailable_without_auxiliary_model"
    )

    # Both local feature forms share the same source metric and availability
    # policy; only their representation differs.
    for feature_name, feature_form in [
        (multimodal_level_column(metric), "origin_level"),
        (multimodal_delta_column(metric), "prior_daypart_change"),
    ]:
        context_catalog_records.append({
            "feature_name": feature_name,
            "feature_family": "multimodal_origin",
            "context_subfamily": "forecast_target_source" if is_target_source else "auxiliary_source",
            "source_metric": metric,
            "source_metric_label": METRIC_LABELS[metric],
            "metric_value_class": context_value_class(metric),
            "feature_form": feature_form,
            "ordinary_availability_class": "observed_through_origin",
            "pre_cp_post_freeze_class": post_freeze_class,
            "counterfactual_runtime_source": runtime_source,

            # Only local target-derived fields duplicate the richer 4.2.1
            # target history. These will be structurally excluded during final
            # target-specific assembly.
            "drop_when_source_metric_equals_target_metric": is_target_source,

            "missing_value_policy": context_missing_policy(metric),
            "requires_cross_zone_recursive_state": False,

            # This field is deliberately explicit: no legal counterfactual
            # feature may silently fall back to observed post-CP mobility.
            "uses_actual_post_cp_mobility": False,
        })


# ---------------------------------------------------------------------
# 2. Transportation-connected origin state
# ---------------------------------------------------------------------

for metric in MOBILITY_CONTEXT_METRICS:
    is_target_source = metric in FORECAST_TARGETS

    context_catalog_records.append({
        "feature_name": connected_feature_column(metric),
        "feature_family": "transportation_connected",
        "context_subfamily": "forecast_target_source" if is_target_source else "auxiliary_source",
        "source_metric": metric,
        "source_metric_label": METRIC_LABELS[metric],
        "metric_value_class": context_value_class(metric),
        "feature_form": "connected_zone_mean",
        "ordinary_availability_class": "observed_through_origin",
        "pre_cp_post_freeze_class": "recursive_required" if is_target_source else "ordinary_only",
        "counterfactual_runtime_source": (
            "recursive_connected_target_state"
            if is_target_source
            else "unavailable_without_auxiliary_model"
        ),

        # A target's connected-zone state is not duplicate local history.
        # It therefore remains eligible even when source metric == target.
        "drop_when_source_metric_equals_target_metric": False,

        "missing_value_policy": context_missing_policy(metric),

        # Reconstructing a connected target feature after the freeze requires
        # recursively predicted states for the neighboring zones as well as
        # the focal zone.
        "requires_cross_zone_recursive_state": is_target_source,
        "uses_actual_post_cp_mobility": False,
    })


# ---------------------------------------------------------------------
# 3. Static geographic and policy context
# ---------------------------------------------------------------------

for feature_name in STATIC_CONTEXT_PREDICTOR_COLUMNS:
    context_catalog_records.append({
        "feature_name": feature_name,
        "feature_family": "static_geography",
        "context_subfamily": "static",
        "source_metric": None,
        "source_metric_label": None,
        "metric_value_class": "static",
        "feature_form": "static_attribute",

        # Static geography is known before, during, and after every forecast
        # origin and does not depend on observed post-CP mobility.
        "ordinary_availability_class": "always_available",
        "pre_cp_post_freeze_class": "static_available",
        "counterfactual_runtime_source": "authoritative_static_taxi_zone_reference",

        "drop_when_source_metric_equals_target_metric": False,
        "missing_value_policy": "Authoritative static value; no temporal imputation.",
        "requires_cross_zone_recursive_state": False,
        "uses_actual_post_cp_mobility": False,
    })


forecast_context_feature_catalog_df = pd.DataFrame(context_catalog_records)

# A feature name must map to exactly one contract row. Duplicate names would
# make downstream feature selection and runtime behavior ambiguous.
if forecast_context_feature_catalog_df["feature_name"].duplicated().any():
    duplicate_names = (
        forecast_context_feature_catalog_df.loc[
            forecast_context_feature_catalog_df["feature_name"].duplicated(False),
            "feature_name",
        ]
        .unique()
        .tolist()
    )
    raise ValueError(f"Context feature catalog contains duplicate names: {duplicate_names}")


# The Pre-CP runtime contract is a deliberately narrower export containing only
# the fields needed when the frozen model is recursively advanced beyond
# January 4, 2025.
pre_cp_context_runtime_contract_df = forecast_context_feature_catalog_df[
    [
        "feature_name", "feature_family", "context_subfamily",
        "source_metric", "pre_cp_post_freeze_class",
        "counterfactual_runtime_source",
        "drop_when_source_metric_equals_target_metric",
        "requires_cross_zone_recursive_state",
        "uses_actual_post_cp_mobility",
    ]
].copy()

pre_cp_context_runtime_contract_df["post_freeze_action"] = (
    pre_cp_context_runtime_contract_df["pre_cp_post_freeze_class"].map({
        "recursive_required": "Recompute from recursively predicted no-CP target mobility.",
        "ordinary_only": "Exclude unless the auxiliary mobility measure is separately modeled.",
        "static_available": "Carry forward unchanged.",
    })
)

if WRITE_FINAL_OUTPUTS:
    forecast_context_feature_catalog_df.to_parquet(FINAL_CONTEXT_CATALOG_PATH, index=False)
    pre_cp_context_runtime_contract_df.to_parquet(PRE_CP_CONTEXT_RUNTIME_CONTRACT_PATH, index=False)


# Reader-facing summary collapses the detailed feature-level contract into the
# five combinations that actually differ in ordinary/counterfactual behavior.
availability_summary_df = (
    forecast_context_feature_catalog_df
    .groupby(
        [
            "feature_family", "context_subfamily",
            "ordinary_availability_class", "pre_cp_post_freeze_class",
        ],
        observed=True,
        dropna=False,
    )
    .agg(
        predictors=("feature_name", "nunique"),
        same_target_drop_features=("drop_when_source_metric_equals_target_metric", "sum"),
        cross_zone_recursive_features=("requires_cross_zone_recursive_state", "sum"),
    )
    .reset_index()
)

availability_summary_df["feature_family"] = availability_summary_df["feature_family"].map({
    "multimodal_origin": "Multimodal origin state",
    "transportation_connected": "Transportation-connected state",
    "static_geography": "Static geography / policy",
})

availability_summary_df["context_subfamily"] = availability_summary_df["context_subfamily"].map({
    "forecast_target_source": "Forecast-target source",
    "auxiliary_source": "Auxiliary source",
    "static": "Static",
})

display(
    availability_summary_df.rename(columns={
        "feature_family": "Feature family",
        "context_subfamily": "Source type",
        "ordinary_availability_class": "Ordinary forecasting",
        "pre_cp_post_freeze_class": "No-CP after Jan. 4",
        "predictors": "Predictors",
        "same_target_drop_features": "Drop for same target",
        "cross_zone_recursive_features": "Cross-zone recursion",
    })
)

,Feature family,Source type,Ordinary forecasting,No-CP after Jan. 4,Predictors,Drop for same target,Cross-zone recursion
0,Multimodal origin state,Auxiliary source,observed_through_origin,ordinary_only,10,0,0
1,Multimodal origin state,Forecast-target source,observed_through_origin,recursive_required,10,10,0
2,Static geography / policy,Static,always_available,static_available,11,0,0
3,Transportation-connected state,Auxiliary source,observed_through_origin,ordinary_only,5,0,0
4,Transportation-connected state,Forecast-target source,observed_through_origin,recursive_required,5,0,5


**Findings.** The final context contract separates the 41 retained predictors into five availability groups with clear ordinary-forecasting and no-CP behavior. All 15 target-derived dynamic features—10 local multimodal level/change fields and 5 transportation-connected target fields—remain usable after the January 4 freeze only through recursive reconstruction, with the 5 connected target features explicitly requiring cross-zone recursive state. The 15 auxiliary dynamic features are observed through ordinary forecast origins but become unavailable to the frozen no-CP simulation unless those auxiliary measures are separately modeled. All 11 static geographic/policy predictors remain temporally available across regimes. The catalog also marks exactly the 10 local target-derived fields for same-target exclusion, while no connected or static feature is removed on that basis.

**Assemble the final 4.2.2 origin-context surfaces.** Attach the compact multimodal state and completed spatial context to each native 4.2.1 Forecast Sequence without changing the inherited modeling populations.

The shared schema contains all 41 selected context columns: 20 local multimodal candidates, 10 transportation-connected candidates, and 11 static geographic/policy candidates. During target-specific assembly, the target's own two local multimodal fields are structurally blanked because 4.2.1 already supplies the richer target-history representation. The remaining 39 context columns stay eligible subject to their ordinary missingness and availability.

Support and missingness fields travel with the surfaces for later QA and model preprocessing, but they are kept outside the predictor catalog.

In [18]:
# 4.2.2.4B — Assemble target-specific origin-context surfaces

# Local multimodal state contributes exactly two columns per approved mobility
# measure: the origin level and exact previous-Daypart change.
MULTIMODAL_CONTEXT_FEATURE_COLUMNS = [
    column
    for metric in MOBILITY_CONTEXT_METRICS
    for column in [multimodal_level_column(metric), multimodal_delta_column(metric)]
]

FINAL_CONTEXT_PREDICTOR_COLUMNS = (
    MULTIMODAL_CONTEXT_FEATURE_COLUMNS
    + SPATIAL_CONNECTED_FEATURE_COLUMNS
    + STATIC_CONTEXT_PREDICTOR_COLUMNS
)

# Missingness/support fields explain whether a feature value was observed,
# structurally zero, or unavailable. Preserve them for QA/preprocessing, but do
# not silently turn them into production predictors.
MULTIMODAL_SUPPORT_METADATA_COLUMNS = [
    column
    for column in multimodal_candidate_state_df.columns
    if column.startswith("mm_")
    and (
        column.endswith("_origin_missing_flag")
        or column.endswith("_seq_delta_1_missing_input_flag")
    )
]

SPATIAL_SUPPORT_METADATA_COLUMNS = [
    column
    for column in spatial_context_origin_state_df.columns
    if column.startswith("spatial_connected_support_")
    or column == "spatial_network_neighbor_count"
]

# If final surfaces already exist, reuse them only when they contain the full
# current contract. This prevents a stale draft export from being mistaken for
# the completed 4.2.2 handoff.
outputs_current = all(
    path.exists()
    and set(FINAL_CONTEXT_PREDICTOR_COLUMNS).issubset(parquet_columns(path))
    for path in FINAL_CONTEXT_SURFACE_PATHS.values()
)

needs_context_rebuild = REBUILD_FINAL_CONTEXT_LAYER or not outputs_current

if needs_context_rebuild and not WRITE_FINAL_OUTPUTS:
    raise RuntimeError(
        "Final 4.2.2 context surfaces need rebuilding, "
        "but WRITE_FINAL_OUTPUTS=False."
    )


if needs_context_rebuild:
    for track, origin_path in ORIGIN_FEATURE_PATHS.items():

        # Read only the 4.2.1 identity/origin backbone. The target's own rich
        # temporal features remain in the separate 4.2.1 surface and will be
        # combined later in 4.2.3.
        origin_df = pd.read_parquet(origin_path, columns=ORIGIN_REQUIRED_COLUMNS)

        # Multimodal context was intentionally built once per physical origin
        # rather than once per forecast target. Select the current modeling
        # track and attach it many-to-one to the target-specific backbone.
        track_mm_df = multimodal_candidate_state_df.loc[
            multimodal_candidate_state_df["modeling_track"].eq(track),
            PHYSICAL_ORIGIN_KEYS
            + MULTIMODAL_CONTEXT_FEATURE_COLUMNS
            + MULTIMODAL_SUPPORT_METADATA_COLUMNS,
        ].copy()

        # Spatial context already contains the connected mobility state plus
        # inherited static geography from 4.2.2.3E.
        spatial_attach_columns = list(dict.fromkeys(
            STATIC_CONTEXT_METADATA_COLUMNS
            + SPATIAL_CONNECTED_FEATURE_COLUMNS
            + STATIC_CONTEXT_PREDICTOR_COLUMNS
            + SPATIAL_SUPPORT_METADATA_COLUMNS
        ))

        track_spatial_df = spatial_context_origin_state_df.loc[
            spatial_context_origin_state_df["modeling_track"].eq(track),
            PHYSICAL_ORIGIN_KEYS + spatial_attach_columns,
        ].copy()

        # Both source objects must remain target-neutral physical-origin
        # surfaces. A duplicate here would multiply forecast-sequence rows
        # during the joins below.
        validate_unique_key(
            track_mm_df,
            PHYSICAL_ORIGIN_KEYS,
            f"{TRACK_LABELS[track]} multimodal context",
        )
        validate_unique_key(
            track_spatial_df,
            PHYSICAL_ORIGIN_KEYS,
            f"{TRACK_LABELS[track]} spatial context",
        )

        # Attach local multimodal state and require every 4.2.1 origin to find
        # exactly one matching physical-origin context row.
        context_df = origin_df.merge(
            track_mm_df,
            on=PHYSICAL_ORIGIN_KEYS,
            how="left",
            validate="many_to_one",
            indicator="_multimodal_join",
        )

        if not context_df["_multimodal_join"].eq("both").all():
            unmatched = int(context_df["_multimodal_join"].ne("both").sum())
            raise ValueError(
                f"{TRACK_LABELS[track]} contains "
                f"{unmatched:,} unmatched multimodal origin rows."
            )

        context_df = context_df.drop(columns="_multimodal_join")

        # Attach dynamic connected-zone state and static geography using the
        # same physical-origin key. Again, no native forecast row may disappear
        # or multiply.
        context_df = context_df.merge(
            track_spatial_df,
            on=PHYSICAL_ORIGIN_KEYS,
            how="left",
            validate="many_to_one",
            indicator="_spatial_join",
        )

        if not context_df["_spatial_join"].eq("both").all():
            unmatched = int(context_df["_spatial_join"].ne("both").sum())
            raise ValueError(
                f"{TRACK_LABELS[track]} contains "
                f"{unmatched:,} unmatched spatial origin rows."
            )

        context_df = context_df.drop(columns="_spatial_join")

        # The target-neutral local multimodal state contains all 10 measures.
        # Once the forecast target is known, remove its own level and Δ1 because
        # 4.2.1 already provides a much richer history for that same series.
        #
        # Do NOT remove spatial_connected__<target>: the connected-zone state
        # describes neighboring mobility and is genuinely additional context.
        for target_metric in FORECAST_TARGETS:
            target_rows = context_df["metric"].eq(target_metric)
            own_local_columns = [
                multimodal_level_column(target_metric),
                multimodal_delta_column(target_metric),
            ]
            context_df.loc[target_rows, own_local_columns] = np.nan

        # Target-specific assembly must preserve the exact 4.2.1 grain:
        # Modeling Track × Forecast Sequence × origin_sequence_id.
        validate_unique_key(
            context_df,
            ORIGIN_JOIN_KEYS,
            f"{TRACK_LABELS[track]} final context surface",
        )

        # Keep identity/provenance first, predictors next, then support metadata.
        # dict.fromkeys preserves order while preventing accidental duplicates
        # such as canonical_location_id appearing through more than one source.
        ordered_columns = list(dict.fromkeys(
            ORIGIN_REQUIRED_COLUMNS
            + STATIC_CONTEXT_METADATA_COLUMNS
            + FINAL_CONTEXT_PREDICTOR_COLUMNS
            + MULTIMODAL_SUPPORT_METADATA_COLUMNS
            + SPATIAL_SUPPORT_METADATA_COLUMNS
        ))

        context_df = context_df[ordered_columns]
        context_df.to_parquet(FINAL_CONTEXT_SURFACE_PATHS[track], index=False)

        del origin_df, track_mm_df, track_spatial_df, context_df


# Build a compact handoff summary from Parquet metadata and keys rather than
# reloading every wide context surface into memory solely for display.
context_surface_summary_records = []

for track, path in FINAL_CONTEXT_SURFACE_PATHS.items():
    key_df = pd.read_parquet(path, columns=ORIGIN_JOIN_KEYS + ["taxi_zone_id"])
    schema_columns = parquet_columns(path)

    context_surface_summary_records.append({
        "modeling_track": TRACK_LABELS[track],
        "context_rows": pq.ParquetFile(path).metadata.num_rows,
        "forecast_sequences": int(key_df["forecast_sequence_id"].nunique()),
        "taxi_zones": int(key_df["taxi_zone_id"].nunique()),
        "context_predictors_in_schema": sum(
            column in schema_columns
            for column in FINAL_CONTEXT_PREDICTOR_COLUMNS
        ),

        # The schema carries all 41 candidates for consistency, but the target's
        # own two local fields are structurally blank for each Forecast Sequence.
        "eligible_context_predictors_per_target":
            len(FINAL_CONTEXT_PREDICTOR_COLUMNS) - 2,

        "duplicate_origin_keys": int(key_df.duplicated(ORIGIN_JOIN_KEYS).sum()),
    })

    del key_df

final_context_surface_summary_df = pd.DataFrame(context_surface_summary_records)
display(final_context_surface_summary_df)

,modeling_track,context_rows,forecast_sequences,taxi_zones,context_predictors_in_schema,eligible_context_predictors_per_target,duplicate_origin_keys
0,Post-CP Primary,2497432,1108,257,41,39,0
1,Full-History Regime-Aware,6231379,1051,258,41,39,0
2,Pre-CP Reference,3818325,1039,258,41,39,0


**Findings.** The assembled 4.2.2 context surfaces exactly preserve the native 4.2.1 forecasting populations: 2,497,432 rows across 1,108 Forecast Sequences for Post-CP Primary, 6,231,379 across 1,051 for Full-History Regime-Aware, and 3,817,286 across 1,039 for Pre-CP Reference. Every surface carries the same 41-column context schema and reports zero duplicate origin keys. After structurally blanking the forecast target's own local level and one-step change, each Forecast Sequence has 39 eligible 4.2.2 context candidates before ordinary feature missingness is considered. The differing Taxi Zone counts—257 Post-CP versus 258 in the two longer-history tracks—are inherited from the native forecasting populations rather than introduced by context assembly.

**Run final end-to-end context QA.** The final checks validate the 4.2.2 handoff as a whole rather than repeating the transformation-level QA already completed above.

The exported surfaces must preserve every native 4.2.1 forecast-origin row, remain unique at the inherited Forecast Sequence × origin grain, contain the complete selected context schema, and reproduce the exact Post-CP and Pre-CP sequence boundaries.

The QA also enforces the less-visible forecasting rules: every scheduled forecast origin must resolve to context, the target's own local multimodal pair must be excluded, auxiliary dynamic context must remain unavailable after the Pre-CP freeze, connected target context must require cross-zone recursive state, support fields must remain metadata, and no weather or actual post-CP counterfactual mobility may enter the 4.2.2 predictor contract.

Only after every check passes is the final handoff manifest written.

In [19]:
# 4.2.2.4C — Run final context-contract QA and write the handoff manifest

def context_contract_qa_row(check, passed, detail):
    """Create one standardized reader-facing QA result."""
    return {
        "check": check,
        "status": "PASS" if bool(passed) else "FAIL",
        "detail": detail,
    }


# ---------------------------------------------------------------------
# 1. Audit each exported context surface against its 4.2.1 backbone
# ---------------------------------------------------------------------

surface_audit_records = []

# These are the 10 local columns that may be structurally blank depending on
# which one of the five target metrics a Forecast Sequence represents.
target_local_context_columns = [
    column
    for metric in FORECAST_TARGETS
    for column in [multimodal_level_column(metric), multimodal_delta_column(metric)]
]

for track in MODELING_TRACKS:
    context_path = FINAL_CONTEXT_SURFACE_PATHS[track]
    origin_path = ORIGIN_FEATURE_PATHS[track]
    context_schema = parquet_columns(context_path)

    # Load only fields needed to validate identity, target-specific exclusion,
    # and sequence boundaries. There is no need to materialize every predictor
    # column again for structural QA.
    context_audit_df = pd.read_parquet(
        context_path,
        columns=list(dict.fromkeys(
            ORIGIN_JOIN_KEYS
            + ["taxi_zone_id", "metric"]
            + target_local_context_columns
        )),
    )

    origin_key_df = pd.read_parquet(origin_path, columns=ORIGIN_JOIN_KEYS)

    # Verify the target-specific rule metric by metric. For example, Taxi-trip
    # Forecast Sequences must have local Taxi-trip level/Δ1 blank, but their
    # FHVHV/Subway/Bus context remains untouched.
    same_target_exclusion_ok = True

    for target_metric in FORECAST_TARGETS:
        target_rows = context_audit_df["metric"].eq(target_metric)
        own_columns = [
            multimodal_level_column(target_metric),
            multimodal_delta_column(target_metric),
        ]

        same_target_exclusion_ok &= (
            context_audit_df.loc[target_rows, own_columns]
            .isna()
            .all()
            .all()
        )

    # Forecast scheduling is horizon-specific, but context is origin-specific.
    # Every origin referenced by the h=1/h=2/h=5 calendar must therefore exist
    # in the corresponding 4.2.2 origin surface.
    scheduled_origin_ids = (
        forecast_target_context_df.loc[
            forecast_target_context_df["modeling_track"].eq(track),
            "origin_observation_sequence_id",
        ]
        .dropna()
        .unique()
    )

    available_origin_ids = (
        context_audit_df["observation_sequence_id"]
        .dropna()
        .unique()
    )

    unresolved_scheduled_origins = int(
        len(np.setdiff1d(scheduled_origin_ids, available_origin_ids))
    )

    surface_audit_records.append({
        "modeling_track": track,
        "context_rows": len(context_audit_df),
        "origin_rows": pq.ParquetFile(origin_path).metadata.num_rows,
        "duplicate_origin_keys": int(context_audit_df.duplicated(ORIGIN_JOIN_KEYS).sum()),
        "context_sequences": int(context_audit_df["forecast_sequence_id"].nunique()),
        "origin_sequences": int(origin_key_df["forecast_sequence_id"].nunique()),
        "context_min_sequence_id": int(context_audit_df["observation_sequence_id"].min()),
        "context_max_sequence_id": int(context_audit_df["observation_sequence_id"].max()),
        "origin_min_sequence_id": int(origin_key_df["observation_sequence_id"].min()),
        "origin_max_sequence_id": int(origin_key_df["observation_sequence_id"].max()),
        "predictor_columns_present": sum(
            column in context_schema
            for column in FINAL_CONTEXT_PREDICTOR_COLUMNS
        ),
        "same_target_exclusion_ok": same_target_exclusion_ok,
        "unresolved_scheduled_origins": unresolved_scheduled_origins,
    })

    del context_audit_df, origin_key_df


surface_audit_df = pd.DataFrame(surface_audit_records)
surface_lookup = surface_audit_df.set_index("modeling_track")


# ---------------------------------------------------------------------
# 2. Validate the feature-level availability/runtime contract
# ---------------------------------------------------------------------

catalog = forecast_context_feature_catalog_df

# These counts encode the approved design itself. If one changes unexpectedly,
# the final QA should fail rather than silently accepting a different feature
# architecture.
expected_family_counts = {
    "multimodal_origin": len(MOBILITY_CONTEXT_METRICS) * 2,          # 20
    "transportation_connected": len(MOBILITY_CONTEXT_METRICS),      # 10
    "static_geography": len(STATIC_CONTEXT_PREDICTOR_COLUMNS),      # 11
}

actual_family_counts = (
    catalog.groupby("feature_family", observed=True)["feature_name"]
    .nunique()
    .to_dict()
)

# Each target contributes two local recursive features plus one connected
# recursive feature. Each auxiliary contributes the same three dynamic forms,
# but those become unavailable after the Pre-CP freeze.
expected_recursive = len(FORECAST_TARGETS) * 3
expected_ordinary_only = len(AUXILIARY_CONTEXT_METRICS) * 3
expected_same_target_drops = len(FORECAST_TARGETS) * 2
expected_cross_zone_recursive = len(FORECAST_TARGETS)


# ---------------------------------------------------------------------
# 3. Assemble one compact executable QA record
# ---------------------------------------------------------------------

final_context_qa_rows = [
    context_contract_qa_row(
        "Final context surfaces preserve every 4.2.1 origin row",
        surface_audit_df["context_rows"].eq(surface_audit_df["origin_rows"]).all(),
        "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.context_rows:,}/{row.origin_rows:,}"
            for row in surface_audit_df.itertuples(index=False)
        ),
    ),

    context_contract_qa_row(
        "Final context origin keys are unique",
        surface_audit_df["duplicate_origin_keys"].eq(0).all(),
        "duplicate rows by track = "
        + "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.duplicate_origin_keys:,}"
            for row in surface_audit_df.itertuples(index=False)
        ),
    ),

    context_contract_qa_row(
        "Native Forecast Sequence populations are preserved",
        surface_audit_df["context_sequences"].eq(surface_audit_df["origin_sequences"]).all(),
        "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.context_sequences:,}/{row.origin_sequences:,} sequences"
            for row in surface_audit_df.itertuples(index=False)
        ),
    ),

    context_contract_qa_row(
        "Context sequence bounds match the 4.2.1 origin surfaces",
        (
            surface_audit_df["context_min_sequence_id"].eq(
                surface_audit_df["origin_min_sequence_id"]
            )
            & surface_audit_df["context_max_sequence_id"].eq(
                surface_audit_df["origin_max_sequence_id"]
            )
        ).all(),
        "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.context_min_sequence_id:,}–{row.context_max_sequence_id:,}"
            for row in surface_audit_df.itertuples(index=False)
        ),
    ),

    context_contract_qa_row(
        "Congestion-pricing track boundary is exact",
        (
            surface_lookup.loc[
                "post_cp_primary",
                "context_min_sequence_id",
            ] == EXPECTED_CP_FIRST_SEQUENCE_ID
            and surface_lookup.loc[
                "pre_cp_reference",
                "context_max_sequence_id",
            ] == EXPECTED_PRE_CP_FREEZE_SEQUENCE_ID
        ),
        (
            f"Post-CP begins "
            f"{surface_lookup.loc['post_cp_primary', 'context_min_sequence_id']:,}; "
            f"Pre-CP ends "
            f"{surface_lookup.loc['pre_cp_reference', 'context_max_sequence_id']:,}"
        ),
    ),

    context_contract_qa_row(
        "Every selected context predictor is present in every surface",
        surface_audit_df["predictor_columns_present"].eq(
            len(FINAL_CONTEXT_PREDICTOR_COLUMNS)
        ).all(),
        f"expected predictor columns per surface = "
        f"{len(FINAL_CONTEXT_PREDICTOR_COLUMNS):,}",
    ),

    context_contract_qa_row(
        "Target's own local multimodal level and change are excluded",
        surface_audit_df["same_target_exclusion_ok"].all(),
        "two redundant local multimodal columns are structurally blank for each target",
    ),

    context_contract_qa_row(
        "Every scheduled forecast origin resolves to the context surfaces",
        surface_audit_df["unresolved_scheduled_origins"].eq(0).all(),
        "unresolved scheduled origin positions by track = "
        + "; ".join(
            f"{TRACK_LABELS[row.modeling_track]} "
            f"{row.unresolved_scheduled_origins:,}"
            for row in surface_audit_df.itertuples(index=False)
        ),
    ),

    context_contract_qa_row(
        "Context feature names are unique",
        not catalog["feature_name"].duplicated().any(),
        f"catalog rows = {len(catalog):,}; "
        f"unique names = {catalog['feature_name'].nunique():,}",
    ),

    context_contract_qa_row(
        "Context family counts match the selected design",
        actual_family_counts == expected_family_counts,
        f"actual = {actual_family_counts}; "
        f"expected = {expected_family_counts}",
    ),

    context_contract_qa_row(
        "Target-derived dynamic context is recursive after the Pre-CP freeze",
        int(
            catalog["pre_cp_post_freeze_class"]
            .eq("recursive_required")
            .sum()
        ) == expected_recursive,
        f"recursive-required features = "
        f"{int(catalog['pre_cp_post_freeze_class'].eq('recursive_required').sum()):,}/"
        f"{expected_recursive:,}",
    ),

    context_contract_qa_row(
        "Auxiliary dynamic context is ordinary-forecasting-only after the freeze",
        int(
            catalog["pre_cp_post_freeze_class"]
            .eq("ordinary_only")
            .sum()
        ) == expected_ordinary_only,
        f"ordinary-only features = "
        f"{int(catalog['pre_cp_post_freeze_class'].eq('ordinary_only').sum()):,}/"
        f"{expected_ordinary_only:,}",
    ),

    context_contract_qa_row(
        "Static context remains available after the freeze",
        int(
            catalog["pre_cp_post_freeze_class"]
            .eq("static_available")
            .sum()
        ) == len(STATIC_CONTEXT_PREDICTOR_COLUMNS),
        f"static-available features = "
        f"{int(catalog['pre_cp_post_freeze_class'].eq('static_available').sum()):,}/"
        f"{len(STATIC_CONTEXT_PREDICTOR_COLUMNS):,}",
    ),

    context_contract_qa_row(
        "Same-target drop flags apply only to intended local target features",
        int(
            catalog["drop_when_source_metric_equals_target_metric"]
            .sum()
        ) == expected_same_target_drops,
        f"same-target drop flags = "
        f"{int(catalog['drop_when_source_metric_equals_target_metric'].sum()):,}/"
        f"{expected_same_target_drops:,}",
    ),

    context_contract_qa_row(
        "Connected target context is marked for cross-zone recursion",
        int(
            catalog["requires_cross_zone_recursive_state"]
            .sum()
        ) == expected_cross_zone_recursive,
        f"cross-zone recursive features = "
        f"{int(catalog['requires_cross_zone_recursive_state'].sum()):,}/"
        f"{expected_cross_zone_recursive:,}",
    ),

    context_contract_qa_row(
        "No counterfactual context feature may read actual post-CP mobility",
        not catalog["uses_actual_post_cp_mobility"].any(),
        "features permitting actual post-CP mobility in the no-CP runtime = 0",
    ),

    context_contract_qa_row(
        "QA/support fields are not promoted into the predictor catalog",
        not catalog["feature_name"].str.contains(
            r"missing_flag|support_",
            case=False,
            regex=True,
            na=False,
        ).any(),
        "support and missingness diagnostics remain metadata only",
    ),

    context_contract_qa_row(
        "Weather is not included in the 4.2.2 context contract",
        not catalog["feature_name"].str.contains(
            "weather",
            case=False,
            regex=False,
            na=False,
        ).any(),
        "weather remains deferred to 4.2.3",
    ),
]

final_context_contract_qa_df = pd.DataFrame(final_context_qa_rows)

if WRITE_FINAL_OUTPUTS:
    final_context_contract_qa_df.to_parquet(FINAL_CONTEXT_QA_PATH, index=False)

display(final_context_contract_qa_df)

# Fail immediately if any structural or leakage-safety contract is violated.
# The manifest below represents an approved handoff and therefore must never be
# written from a partially failing feature layer.
if not final_context_contract_qa_df["status"].eq("PASS").all():
    failed_checks = (
        final_context_contract_qa_df.loc[
            final_context_contract_qa_df["status"].eq("FAIL"),
            "check",
        ]
        .tolist()
    )

    raise AssertionError(f"Final 4.2.2 context QA failed: {failed_checks}")


# ---------------------------------------------------------------------
# 4. Write the handoff manifest only after every final QA check passes
# ---------------------------------------------------------------------

if WRITE_FINAL_OUTPUTS:
    manifest_specs = [
        (
            f"Origin context surface · {TRACK_LABELS[track]}",
            path,
            "Target-specific multimodal, spatial, and static origin context",
        )
        for track, path in FINAL_CONTEXT_SURFACE_PATHS.items()
    ] + [
        (
            "Forecast context feature catalog",
            FINAL_CONTEXT_CATALOG_PATH,
            "Predictor definitions and ordinary/counterfactual availability rules",
        ),
        (
            "Pre-CP context runtime contract",
            PRE_CP_CONTEXT_RUNTIME_CONTRACT_PATH,
            "Post-freeze recursive, static, and unavailable feature behavior",
        ),
        (
            "Final context contract QA",
            FINAL_CONTEXT_QA_PATH,
            "End-to-end structural and leakage-safety validation",
        ),
    ]

    manifest_records = []

    for artifact, path, role in manifest_specs:
        parquet_file = pq.ParquetFile(path)

        manifest_records.append({
            "artifact": artifact,
            "path": str(path),
            "rows": parquet_file.metadata.num_rows,
            "columns": len(parquet_file.schema_arrow.names),
            "role": role,
        })

    forecast_context_manifest_df = pd.DataFrame(manifest_records)
    forecast_context_manifest_df.to_parquet(FINAL_CONTEXT_MANIFEST_PATH, index=False)

,check,status,detail
0,Final context surfaces preserve every 4.2.1 origin row,PASS,"Post-CP Primary 2,497,432/2,497,432; Full-History Regime-Aware 6,231,379/6,231,379; Pre-CP Reference 3,818,325/3,818,325"
1,Final context origin keys are unique,PASS,duplicate rows by track = Post-CP Primary 0; Full-History Regime-Aware 0; Pre-CP Reference 0
2,Native Forecast Sequence populations are preserved,PASS,"Post-CP Primary 1,108/1,108 sequences; Full-History Regime-Aware 1,051/1,051 sequences; Pre-CP Reference 1,039/1,039 sequences"
3,Context sequence bounds match the 4.2.1 origin surfaces,PASS,"Post-CP Primary 3,676–5,929; Full-History Regime-Aware 1–5,929; Pre-CP Reference 1–3,675"
4,Congestion-pricing track boundary is exact,PASS,"Post-CP begins 3,676; Pre-CP ends 3,675"
5,Every selected context predictor is present in every surface,PASS,expected predictor columns per surface = 41
6,Target's own local multimodal level and change are excluded,PASS,two redundant local multimodal columns are structurally blank for each target
7,Every scheduled forecast origin resolves to the context surfaces,PASS,unresolved scheduled origin positions by track = Post-CP Primary 0; Full-History Regime-Aware 0; Pre-CP Reference 0
8,Context feature names are unique,PASS,catalog rows = 41; unique names = 41
9,Context family counts match the selected design,PASS,"actual = {'multimodal_origin': 20, 'static_geography': 11, 'transportation_connected': 10}; expected = {'multimodal_origin': 20, 'transportation_connected': 10, 'static_geography': 11}"


**Findings.** All 18 final context-contract checks pass, closing the 4.2.2 handoff without changing any native forecasting population or sequence boundary. The three surfaces preserve all 2,497,432 Post-CP, 6,231,379 Full-History, and 3,817,286 Pre-CP origin rows with zero duplicate keys; all scheduled forecast origins resolve, the Post-CP/Pre-CP boundary remains exactly 3,676/3,675, and all 41 selected context columns are present. The target-specific exclusions and runtime rules also validate as designed: 10 same-target local fields are flagged for exclusion, 15 target-derived dynamic features are recursive after the freeze, 15 auxiliary dynamic features are ordinary-only, 5 connected target features require cross-zone recursion, and all 11 static predictors remain temporally available. No counterfactual feature is permitted to read actual post-CP mobility, support fields remain metadata, and weather remains excluded for 4.2.3.

### Section Summary

This section converts the selected 4.2.2 feature families into an explicit forecasting handoff at the same **Modeling Track × Forecast Sequence × Origin** grain established in 4.2.1.

The final context contract contains **41 shared candidate columns**: 20 local multimodal features, 10 transportation-connected features, and 11 static geographic/policy features. Because the local multimodal state was constructed once in target-neutral form, each Forecast Sequence structurally excludes the two local features derived from its own target metric; 4.2.1 remains responsible for that series' richer history. Connected-zone versions remain eligible because they represent surrounding transportation conditions rather than duplicate local history.

The accompanying runtime contract also makes the counterfactual information boundary explicit. Target-derived dynamic context can be reconstructed from recursively predicted no-CP mobility after the January 4 freeze, including connected-zone context that requires predicted states across multiple Taxi Zones. Auxiliary dynamic context cannot use actual post-CP observations and becomes unavailable unless separately modeled, while static geography remains available throughout.

These exports are still **origin-context surfaces**, not the final forecasting matrices. 4.2.3 will add weather, combine the complete feature universe, align predictors with h=1, h=2, and h=5 targets, render the complete Feature Availability / Feature Family Matrix, and perform the final cross-family leakage QA.

## Close

This notebook extends the target-history and seasonal feature layer from 4.2.1 with a compact representation of the broader transportation system available at each forecast origin.

The completed context design combines three pieces: local multimodal state across 10 mobility measures, transportation-connected conditions across the same measures, and stable geographic and congestion-pricing context from the authoritative Taxi Zone reference. The feature catalog keeps those candidate predictors separate from support and missingness metadata, while the Pre-CP runtime contract records which dynamic features can be rebuilt recursively, which auxiliary features become unavailable after the January 4 freeze, and which static features remain available throughout.

The final outputs preserve the native modeling-track and Forecast Sequence populations established upstream and remain origin-side context surfaces rather than complete model matrices. This notebook intentionally does not add weather or attach h=1, h=2, and h=5 target rows.

Notebook 4.2.3 will complete the forecasting feature layer by adding weather context, combining the 4.2.1 and 4.2.2 predictor families, aligning them to the forecast targets and horizons, visualizing feature-family availability across the forecasting designs, and running the final cross-family leakage and handoff QA.

### 4.2.2 Final Inventory of Files

This notebook writes the following downstream handoff assets to `pipeline_data/4.2.2.final_tables/`. Reusable intermediate/cache artifacts are not included in this final inventory.

- **`origin_context_surface__post_cp_primary.parquet`**  
  Target-specific origin-context surface for the **Post-CP Primary** forecasting track. Carries the selected multimodal, transportation-connected spatial, and static geographic/policy context at the same forecasting grain established in 4.2.1.

- **`origin_context_surface__full_history_regime_aware.parquet`**  
  Target-specific origin-context surface for the **Full-History Regime-Aware** track, spanning the full historical forecasting period while preserving the regime-aware forecasting contract.

- **`origin_context_surface__pre_cp_reference.parquet`**  
  Target-specific observed-history context surface for the **Pre-CP Reference** track through the January 4, 2025 freeze boundary. Post-freeze no-CP context must subsequently follow the recursive runtime contract rather than use actual post-CP mobility observations.

- **`forecast_context_feature_catalog.parquet`**  
  Feature-level catalog for the 4.2.2 context layer. Records each predictor's feature family, source metric, representation, missing-value policy, ordinary-forecast availability, Pre-CP post-freeze availability, same-target exclusion rule, and cross-zone recursive-state requirements.

- **`pre_cp_context_runtime_contract.parquet`**  
  Runtime contract for reconstructing the 4.2.2 context layer after the Pre-CP model is frozen. Identifies target-derived features that can be rebuilt recursively, auxiliary mobility features that become unavailable unless separately forecast, and static features that remain available throughout the no-CP simulation.

- **`final_context_contract_qa.parquet`**  
  Final structural, availability, alignment, and leakage-safety QA results for the completed 4.2.2 context package.

- **`forecast_context_manifest.parquet`**  
  File-level manifest for the 4.2.2 downstream handoff package, recording the exported artifacts and their roles for later forecasting notebooks.


### 4.2.2 Feature Columns

The final 4.2.2 context contract carries **41 candidate predictor columns**. Of these, **30 are newly engineered in 4.2.2** and **11 are inherited static geographic/policy predictors attached from the existing spatial context layer**.

#### Newly Engineered Multimodal Origin Context — 20 Columns

Each of the 10 retained mobility measures contributes its value at the forecast origin and its change from the immediately preceding canonical Daypart.

- `mm_taxi_trip_count_origin_value`
- `mm_taxi_trip_count_seq_delta_1`
- `mm_taxi_avg_trip_speed_origin_value`
- `mm_taxi_avg_trip_speed_seq_delta_1`
- `mm_fhvhv_trip_count_origin_value`
- `mm_fhvhv_trip_count_seq_delta_1`
- `mm_fhvhv_avg_trip_speed_origin_value`
- `mm_fhvhv_avg_trip_speed_seq_delta_1`
- `mm_subway_ridership_origin_value`
- `mm_subway_ridership_seq_delta_1`
- `mm_taxi_avg_trip_duration_origin_value`
- `mm_taxi_avg_trip_duration_seq_delta_1`
- `mm_fhvhv_avg_trip_duration_origin_value`
- `mm_fhvhv_avg_trip_duration_seq_delta_1`
- `mm_subway_transfers_origin_value`
- `mm_subway_transfers_seq_delta_1`
- `mm_bus_trip_count_origin_value`
- `mm_bus_trip_count_seq_delta_1`
- `mm_avg_bus_speed_origin_value`
- `mm_avg_bus_speed_seq_delta_1`

#### Newly Engineered Transportation-Connected Spatial Context — 10 Columns

Each retained mobility measure also contributes the same-origin state of the Taxi Zone's transportation-connected geographic environment.

- `spatial_connected__taxi_trip_count`
- `spatial_connected__taxi_avg_trip_speed`
- `spatial_connected__fhvhv_trip_count`
- `spatial_connected__fhvhv_avg_trip_speed`
- `spatial_connected__subway_ridership`
- `spatial_connected__taxi_avg_trip_duration`
- `spatial_connected__fhvhv_avg_trip_duration`
- `spatial_connected__subway_transfers`
- `spatial_connected__bus_trip_count`
- `spatial_connected__avg_bus_speed`

#### Inherited Static Geographic and Policy Context — 11 Columns

These predictors were engineered upstream and are attached in 4.2.2 rather than newly derived here.

- `borough`
- `is_cbd_zone`
- `is_cbd_gateway_zone`
- `is_cbd_adjacent_zone`
- `cbd_spatial_category`
- `distance_to_cbd_miles`
- `distance_to_gateway_miles`
- `connected_neighbor_count`
- `network_betweenness_centrality`
- `network_closeness_centrality`
- `network_clustering_coefficient`

#### Target-Specific Predictor Count

The common surface schema carries all **41 context columns** for consistency. For each forecast target, however, that target's own local multimodal origin level and one-step change are structurally excluded because 4.2.1 already supplies its richer target-history representation.

Therefore:

**41 shared context columns − 2 redundant same-target multimodal columns = 39 eligible 4.2.2 context predictors per forecast target.**

Missingness flags, connected-zone support counts/shares, network-support fields, identifiers, and other provenance columns are retained where needed for QA or downstream preprocessing but are **not counted as forecasting predictors** in this inventory.

In [21]:
# THROWAWAY VALIDATION — verify the Pre-CP final-refit context surface.

pre_cp_context_path = (
    FINAL_DIR / "origin_context_surface__pre_cp_reference.parquet"
)

validation_rows = []


def record_check(check, passed, detail):
    """Collect one compact validation result."""
    validation_rows.append({
        "Check": check,
        "Status": "PASS" if bool(passed) else "FAIL",
        "Detail": detail,
    })


# ---------------------------------------------------------------------
# Read only the 24 origins needed by the Jan. 1–4 final-refit contract
# ---------------------------------------------------------------------

required_origin_ids = list(range(3651, 3675))

refit_context_df = pd.read_parquet(
    pre_cp_context_path,
    columns=[
        "metric",
        "forecast_sequence_id",
        "observation_sequence_id",
    ],
    filters=[
        ("observation_sequence_id", ">=", min(required_origin_ids)),
        ("observation_sequence_id", "<=", max(required_origin_ids)),
    ],
)


# ---------------------------------------------------------------------
# Summarize coverage by target metric
# ---------------------------------------------------------------------

context_metric_summary_df = (
    refit_context_df
    .groupby("metric", as_index=False)
    .agg(
        forecast_sequences=("forecast_sequence_id", "nunique"),
        rows=("forecast_sequence_id", "size"),
        origin_count=("observation_sequence_id", "nunique"),
        first_origin=("observation_sequence_id", "min"),
        last_origin=("observation_sequence_id", "max"),
    )
)

context_metric_summary_df["origins_per_sequence"] = (
    context_metric_summary_df["rows"]
    / context_metric_summary_df["forecast_sequences"]
)

display(context_metric_summary_df)


# ---------------------------------------------------------------------
# Validate complete 24-origin coverage
# ---------------------------------------------------------------------

sequence_origin_counts_df = (
    refit_context_df
    .groupby(
        ["metric", "forecast_sequence_id"],
        as_index=False,
    )
    .agg(
        origin_count=("observation_sequence_id", "nunique")
    )
)

incomplete_sequences_df = sequence_origin_counts_df.loc[
    sequence_origin_counts_df["origin_count"].ne(24)
].copy()

record_check(
    "Every Pre-CP Forecast Sequence contains all 24 required context origins",
    incomplete_sequences_df.empty,
    f"incomplete sequences = {len(incomplete_sequences_df):,}",
)


# ---------------------------------------------------------------------
# Validate the restored Dec. 31 Evening origin
# ---------------------------------------------------------------------

origin_3655_df = refit_context_df.loc[
    refit_context_df["observation_sequence_id"].eq(3655)
].copy()

native_sequence_count = refit_context_df[
    "forecast_sequence_id"
].nunique()

record_check(
    "Origin 3655 exists for every native Forecast Sequence",
    (
        len(origin_3655_df) == native_sequence_count
        and origin_3655_df["forecast_sequence_id"].nunique()
        == native_sequence_count
    ),
    (
        f"rows at 3655 = {len(origin_3655_df):,}; "
        f"native sequences = {native_sequence_count:,}"
    ),
)

record_check(
    "Pre-CP native Forecast Sequence population remains 1,039",
    native_sequence_count == 1_039,
    f"Forecast Sequences = {native_sequence_count:,}",
)


# ---------------------------------------------------------------------
# Validate expected metric-specific populations
# ---------------------------------------------------------------------

expected_metric_sequences = {
    "taxi_trip_count": 258,
    "taxi_avg_trip_speed": 110,
    "fhvhv_trip_count": 258,
    "fhvhv_avg_trip_speed": 257,
    "subway_ridership": 156,
}

actual_metric_sequences = (
    refit_context_df
    .groupby("metric")["forecast_sequence_id"]
    .nunique()
    .to_dict()
)

record_check(
    "Metric-specific Forecast Sequence populations remain unchanged",
    actual_metric_sequences == expected_metric_sequences,
    str(actual_metric_sequences),
)


# ---------------------------------------------------------------------
# Validate no duplicate sequence × origin rows
# ---------------------------------------------------------------------

duplicate_rows = int(
    refit_context_df.duplicated(
        ["forecast_sequence_id", "observation_sequence_id"]
    ).sum()
)

record_check(
    "Required-origin context slice has no duplicate sequence × origin rows",
    duplicate_rows == 0,
    f"duplicate rows = {duplicate_rows:,}",
)


# ---------------------------------------------------------------------
# Result
# ---------------------------------------------------------------------

throwaway_422_validation_df = pd.DataFrame(validation_rows)
display(throwaway_422_validation_df)

failed_validation_df = throwaway_422_validation_df.loc[
    throwaway_422_validation_df["Status"].eq("FAIL")
]

if not failed_validation_df.empty:
    raise AssertionError(
        "4.2.2 final-refit context validation failed:\n\n"
        + failed_validation_df.to_string(index=False)
    )

print(
    "\n✅ PASS — 4.2.2 now contains the complete Pre-CP "
    "final-refit context surface, including sequence 3655."
)

,metric,forecast_sequences,rows,origin_count,first_origin,last_origin,origins_per_sequence
0,fhvhv_avg_trip_speed,257,6168,24,3651,3674,24.000
1,fhvhv_trip_count,258,6192,24,3651,3674,24.000
2,subway_ridership,156,3744,24,3651,3674,24.000
3,taxi_avg_trip_speed,110,2640,24,3651,3674,24.000
4,taxi_trip_count,258,6192,24,3651,3674,24.000


,Check,Status,Detail
0,Every Pre-CP Forecast Sequence contains all 24 required context origins,PASS,incomplete sequences = 0
1,Origin 3655 exists for every native Forecast Sequence,PASS,"rows at 3655 = 1,039; native sequences = 1,039"
2,"Pre-CP native Forecast Sequence population remains 1,039",PASS,"Forecast Sequences = 1,039"
3,Metric-specific Forecast Sequence populations remain unchanged,PASS,"{'fhvhv_avg_trip_speed': 257, 'fhvhv_trip_count': 258, 'subway_ridership': 156, 'taxi_avg_trip_speed': 110, 'taxi_trip_count': 258}"
4,Required-origin context slice has no duplicate sequence × origin rows,PASS,duplicate rows = 0



✅ PASS — 4.2.2 now contains the complete Pre-CP final-refit context surface, including sequence 3655.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>